# Evidence KG Builder × OpenRouter 全链路调试 Notebook（并发版）

这个 Notebook 用来**一步步测试当前 `evidence-kg-builder` Skill**，并直接调用 OpenRouter 模型：

`stealth/space-bunny-alpha`

它不是只看最终答案，而是把整条链路拆开，让你能分别看到：

```text
原始 PDF / DOCX / XLSX / PPTX / 图片
        ↓
Parse：统一成 DocumentUnits
        ↓
Prepare：Python 先做确定性结构抽取与任务分流
        ↓
Extraction：OpenRouter 并发语义抽取
        ↓
Alignment：OpenRouter 并发实体对齐
        ↓
Build：Canonical Facts + Sparse KG
        ↓
Lean GraphRAG：Structured / Fact / Entity Profile / Graph / Raw 多路召回
        ↓
OpenRouter 最终回答
```

### 这个 Notebook 特别适合看什么？

1. **OpenRouter 到底慢在哪里**：每个请求都会记录 wall time、token、失败/重试情况。
2. **Extraction 模型到底抽出了什么**：完整原始模型 JSON 会保存到磁盘，也会在 Notebook 里显示预览。
3. **Alignment 到底合并了什么**：每轮会显示候选数量、模型决策和最终 canonical entity。
4. **Q2 一类召回问题到底卡在哪层**：可以直接查看 `query_plan`、`entity_profiles`、`structured_hits`、`fact_hits`、`graph paths`、`raw units` 和最终 context。
5. **并发是否真的生效**：Extraction 默认 8 路并发、Alignment 默认 8 路并发、批量 QA 默认 4 路并发。

> 这份 Notebook 不把 API Key 写入文件。Key 只通过 `getpass()` 放在当前 Kernel 内存里。

## Cell 1 — 全局配置

分步构图先运行本格，再运行 Cell 2。仅查询已有图谱可直接运行 Cell 20、22 或 24。

- `PARSER_BACKEND` 可选 `native` 或 `markitdown`；当前为 native。
- `WORK_DIR` 是构图目录；问答不绑定该目录，而是在 `RAG_SEARCH_ROOT` 下自动寻找同后端最新、完整可用的图谱。
- “最新”按 `04_knowledge_graph.json` 修改时间判断；跳过 archive、未完成、损坏或 fingerprint 不一致的图谱。
- 找不到可用新图时，使用 `RAG_FALLBACK_DIR` 中之前 native/openpyxl 图谱的原样副本。
- Cell 20、22、24 均打印实际路径、latest/fallback、后端及节点/事实数量。此过程只读取图谱，不会自动构图。


In [ ]:
from pathlib import Path

PARSER_BACKEND = 'native'

# 强制禁用旧 skill.zip，直接测试当前 Git 工作区源码
SKILL_ZIP = Path('./__disabled_skill__.zip').resolve()
SKILL_SRC = Path('./skill').resolve()

MATERIALS_ROOT = Path('./materials').resolve()

# 构图/日志工作目录；仅问答使用下方独立的 RAG_WORK_DIR
RUN_ROOT = Path('./kg_debug_run_q5q10_fix').resolve()

SKILL_UNPACK_DIR = RUN_ROOT / 'skill_runtime'
WORK_DIR = RUN_ROOT / ('work' if PARSER_BACKEND == 'native' else 'work_markitdown')
MODEL_OUTPUT_DIR = RUN_ROOT / 'model_outputs'
MANIFEST_PATH = RUN_ROOT / 'inputs.json'

MODEL = 'stealth/space-bunny-alpha'
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'

EXTRACT_REASONING = 'medium'
ALIGN_REASONING = 'medium'
QA_REASONING = 'medium'

EXTRACT_CONCURRENCY = 8
ALIGN_CONCURRENCY = 8
QA_CONCURRENCY = 4

EXTRACT_BATCH_SIZE = 5
EXTRACT_CHAR_BUDGET = 14_000
ALIGN_BATCH_SIZE = 8
# 后续分步 cell 和交互入口都需要这些参数。
EXTRACT_MAX_TOKENS = 10_000
ALIGN_MAX_TOKENS = 3_000
QA_MAX_TOKENS = 4_096
RAW_PREVIEW_CHARS = 5_000
CONTEXT_PREVIEW_CHARS = 8_000
HTTP_TIMEOUT = 300
MAX_NETWORK_RETRIES = 2

# 自动寻找当前解析后端最新的完整图谱；旧 native 图谱仅作 fallback。
RAG_SEARCH_ROOT = Path('.').resolve()
RAG_FALLBACK_DIR = Path('./examples/prebuilt/work').resolve()
RAG_WORK_DIR = None  # 由问答入口自动解析；不绑定某个运行目录。

if SKILL_ZIP.is_file():
    SKILL_SOURCE, SKILL_SOURCE_KIND = SKILL_ZIP, 'zip'
elif SKILL_SRC.is_dir():
    SKILL_SOURCE, SKILL_SOURCE_KIND = SKILL_SRC, 'dir'
else:
    SKILL_SOURCE, SKILL_SOURCE_KIND = None, 'missing'

RUN_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('SKILL_SOURCE =', SKILL_SOURCE, f'({SKILL_SOURCE_KIND})')
print('构图 WORK_DIR =', WORK_DIR)
print('问答图谱 = 自动选择最新完整图谱；fallback =', RAG_FALLBACK_DIR)


## Cell 2 — 定位 Skill 并安装依赖

这一格做两件事：

1. 定位 Skill 目录：如果 Cell 1 选中了 `skill.zip` 就解压到本次运行目录；如果是已解压的 `./skill/` 就直接用，不做任何复制；
2. 安装 Skill 自己声明的 `requirements.txt`。

运行后会打印：

- Skill 根目录；
- 关键脚本是否存在；
- pip 安装结果。

如果这里失败，后面先不要继续，因为后续所有 Parse / Prepare / Build 都直接调用 Skill 自带脚本。

In [10]:
import os, sys, shutil, subprocess, json

if 'SKILL_SRC' not in globals() or 'SKILL_SOURCE' not in globals():
    raise RuntimeError('请先运行 Cell 1（全局配置），再运行 Cell 2。')
# Cell 1 的路径也可在内核中手动调整，定位时重新检查实际文件。
SKILL_SOURCE = SKILL_ZIP if SKILL_ZIP.is_file() else SKILL_SRC if SKILL_SRC.is_dir() else None
SKILL_SOURCE_KIND = 'zip' if SKILL_SOURCE == SKILL_ZIP else 'dir' if SKILL_SOURCE is not None else 'missing'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert SKILL_SOURCE is not None, (
    f'既找不到 {SKILL_ZIP}，也找不到已解压目录 {SKILL_SRC}'
)

if SKILL_SOURCE_KIND == 'zip':
    if SKILL_UNPACK_DIR.exists():
        shutil.rmtree(SKILL_UNPACK_DIR)
    SKILL_UNPACK_DIR.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(SKILL_SOURCE), str(SKILL_UNPACK_DIR))
    search_root = SKILL_UNPACK_DIR
else:
    # 已经解压好，直接用，不再复制/解压
    search_root = SKILL_SOURCE

def find_skill_dir(root):
    """root 本身或它的单层子目录里，唯一包含 SKILL.md 的那个 Skill 根目录。"""
    if (root / 'SKILL.md').exists():
        return root
    hits = [p for p in root.iterdir() if p.is_dir() and (p / 'SKILL.md').exists()]
    if len(hits) == 1:
        return hits[0]
    raise AssertionError(
        f'在 {root} 里找不到唯一的 Skill 目录（{len(hits)} 个候选：{hits}）'
    )

SKILL_DIR = find_skill_dir(search_root)
SCRIPTS_DIR = SKILL_DIR / 'scripts'
REQUIREMENTS = SKILL_DIR / 'requirements.txt'

print('SKILL_DIR =', SKILL_DIR)
print('scripts:')
for p in sorted(SCRIPTS_DIR.glob('*.py')):
    print('  -', p.name)

if REQUIREMENTS.exists():
    print('\n安装/确认依赖...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQUIREMENTS)], check=True)
    print('依赖安装完成。')
else:
    print('未发现 requirements.txt，跳过。')
assert PARSER_BACKEND in {'native', 'markitdown'}, '请选择 native 或 markitdown'
if PARSER_BACKEND == 'markitdown':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SKILL_DIR / 'requirements-markitdown.txt')], check=True)


SKILL_DIR = C:\Users\Agesen\Downloads\Task4\github-release\evidence-kg-graphrag\skill\evidence-kg-builder
scripts:
  - export_enriched.py
  - kg_core.py
  - kg_pipeline.py
  - lean_graphrag.py
  - parse_documents.py
  - parse_markitdown.py

安装/确认依赖...
依赖安装完成。


## Cell 3 — 查看材料并定义 `corpus / questions / reference`

Skill 的一个重要原则是：**只有 `corpus` 文件能进入知识图谱**。

- `corpus`：真正用于构图的 PDF / Word / Excel / PPT / 图片；
- `questions`：测试题，只能用于提问，不能当事实来源；
- `reference`：任务书、说明文档等，也不能进入 KG。

下面先扫描目录。下一格用 `ROLE_OVERRIDES` 指定非 corpus 文件。

如果你的 `materials` 目录里本来就只放了五份业务语料，那么 `ROLE_OVERRIDES = {}` 即可。

In [11]:
SUPPORTED_SUFFIXES = {'.pdf', '.docx', '.xlsx', '.pptx', '.png', '.jpg', '.jpeg', '.webp', '.md', '.txt'}

assert MATERIALS_ROOT.exists(), f'材料目录不存在: {MATERIALS_ROOT}'
material_files = sorted([
    p for p in MATERIALS_ROOT.rglob('*')
    if p.is_file() and not p.name.startswith('~$') and p.suffix.lower() in SUPPORTED_SUFFIXES
])

print(f'发现 {len(material_files)} 个支持的文件：')
for i, p in enumerate(material_files):
    print(f'[{i:02d}] {p.relative_to(MATERIALS_ROOT)}')

发现 6 个支持的文件：
[00] 测试集A_产品架构图.png
[01] 测试集A_产品规格与定价文档.pdf
[02] 测试集A_产品运营分析报告.pptx
[03] 测试集A_分析问题集.docx
[04] 测试集A_工作室运营手册.docx
[05] 测试集A_运营数据记录.xlsx


## Cell 4 — 生成输入 Manifest

本测试集已将 `测试集A_分析问题集.docx` 标为 `questions`；其余文件默认为 `corpus`。问题集只用于测试问答，不进入图谱。

如果目录里还有测试问题或任务说明，请在 `ROLE_OVERRIDES` 里改掉。例如：

```python
ROLE_OVERRIDES = {
    'questions.docx': 'questions',
    'task.pdf': 'reference',
}
```

你也可以用相对路径，例如 `subdir/questions.docx`。

运行后请认真看打印出来的表，确认没有把问题集塞进 `corpus`。

In [12]:
# 问题集不能进入构图语料；其他测试题/任务书也请在此指定角色。
ROLE_OVERRIDES = {
    '测试集A_分析问题集.docx': 'questions',
    # 'questions.docx': 'questions',
    # 'task.pdf': 'reference',
}

entries = []
for p in material_files:
    rel = p.relative_to(MATERIALS_ROOT).as_posix()
    role = ROLE_OVERRIDES.get(rel, 'corpus')
    assert role in {'corpus', 'questions', 'reference'}
    assert '问题集' not in p.stem or role != 'corpus', f'问题集不能标为 corpus: {rel}'
    entries.append({'path': rel, 'role': role})

manifest = {'root': str(MATERIALS_ROOT), 'files': entries}
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print('\n角色统计：')
from collections import Counter
print(Counter(x['role'] for x in entries))

{
  "root": "C:\\Users\\Agesen\\Downloads\\Task4\\github-release\\evidence-kg-graphrag\\materials",
  "files": [
    {
      "path": "测试集A_产品架构图.png",
      "role": "corpus"
    },
    {
      "path": "测试集A_产品规格与定价文档.pdf",
      "role": "corpus"
    },
    {
      "path": "测试集A_产品运营分析报告.pptx",
      "role": "corpus"
    },
    {
      "path": "测试集A_分析问题集.docx",
      "role": "questions"
    },
    {
      "path": "测试集A_工作室运营手册.docx",
      "role": "corpus"
    },
    {
      "path": "测试集A_运营数据记录.xlsx",
      "role": "corpus"
    }
  ]
}

角色统计：
Counter({'corpus': 5, 'questions': 1})


## Cell 5 — CLI 辅助函数

当前 Skill 是一个**文件式协议**：Python 负责 parse / prepare / accept / build；模型只负责 extraction / alignment / final answer。

这个辅助函数统一调用 Skill 的 CLI，并把 stdout 当 JSON 读回来。

如果某一步报错，你会直接看到 Skill 返回的 `error` 和 `error_type`，不会悄悄吞掉。

In [13]:
import subprocess, json, shlex, time
from IPython.display import display, JSON, Markdown

PIPELINE = SCRIPTS_DIR / 'kg_pipeline.py'

def run_cli(*args, check=True):
    cmd = [sys.executable, str(PIPELINE), *map(str, args)]
    proc = subprocess.run(cmd, cwd=str(SKILL_DIR), capture_output=True, text=True, encoding='utf-8')
    stdout = proc.stdout.strip()
    stderr = proc.stderr.strip()
    try:
        data = json.loads(stdout) if stdout else {}
    except Exception:
        data = {'raw_stdout': stdout}
    if check and proc.returncode != 0:
        print('COMMAND:', ' '.join(shlex.quote(x) for x in cmd))
        print('STDOUT:', stdout)
        print('STDERR:', stderr)
        raise RuntimeError(f'CLI failed with code {proc.returncode}')
    return data

## Cell 6 — Fresh Parse：把五种文档统一成 `DocumentUnit`

这里一定使用 `--fresh`，避免复用旧版本的 101 个 Unit 或旧 checkpoint。

### 你应该关注的输出

- `unit_count`：最终切成多少个 DocumentUnit；
- `documents[].status`：每种格式是否成功；
- `cached`：fresh 模式应为 false；
- `parser_versions`：实际使用的解析器版本。

Parse 阶段**不调用 OpenRouter**，所以这里没有模型并发问题。

In [14]:
t0 = time.perf_counter()
if PARSER_BACKEND == 'native':
    parse_result = run_cli('parse', '--manifest', MANIFEST_PATH, '--work', WORK_DIR, '--fresh')
else:
    proc = subprocess.run([sys.executable, str(SCRIPTS_DIR / 'parse_markitdown.py'),
                           '--manifest', str(MANIFEST_PATH), '--work', str(WORK_DIR), '--fresh'],
                          cwd=str(SKILL_DIR), capture_output=True, text=True, encoding='utf-8')
    if proc.returncode != 0:
        print(proc.stdout, proc.stderr)
        raise RuntimeError('MarkItDown parse failed; see the JSON error above')
    parse_result = json.loads(proc.stdout)
parse_elapsed = time.perf_counter() - t0

display(JSON(parse_result))
print(f'\nParse wall time: {parse_elapsed:.2f}s')

<IPython.core.display.JSON object>


Parse wall time: 2.64s


## Cell 7 — 看 `DocumentUnit` 到底长什么样

`00_document_units.json` 是所有后续工作的底座。

每个 Unit 主要包含：

- `format`：pdf/docx/xlsx/pptx/image；
- `locator`：页码、Sheet、行、Slide、Section 等；
- `content_md`：给 LLM 看的紧凑阅读视图；
- `structured`：机器可读的表格/shape/cell ledger；
- `asset_refs`：需要视觉模型查看的图片资产。

这一格会显示格式分布，并抽几个 Unit 给你看。若这里就丢内容，后面再强的模型也救不回来。

In [15]:
import pandas as pd

parsed = json.loads((WORK_DIR / '00_document_units.json').read_text(encoding='utf-8'))
units = parsed['units']
docs = {d['document_id']: d for d in parsed['documents']}

summary_rows = []
for u in units:
    summary_rows.append({
        'unit_id': u['unit_id'],
        'format': u.get('format'),
        'source': docs[u['document_id']]['path'],
        'locator': json.dumps(u.get('locator', {}), ensure_ascii=False),
        'chars': len(u.get('content_md', '')),
        'assets': len(u.get('asset_refs', [])),
        'structured_kind': (u.get('structured') or {}).get('kind'),
    })
unit_df = pd.DataFrame(summary_rows)
display(unit_df.groupby(['format', 'structured_kind'], dropna=False).size().rename('count').reset_index())
display(unit_df.head(12))

print('\n--- 第一个 Unit 的 content_md 预览 ---')
print(units[0].get('content_md', '')[:3000])
print('\nlocator =', units[0].get('locator'))
print('asset_refs =', units[0].get('asset_refs'))

,format,structured_kind,count
0,docx,source_xml_table,2
1,docx,NaN,13
2,pdf,pdf_page,16
3,pdf,pdf_table,10
4,png,NaN,1
5,pptx,pptx_shapes,8
6,xlsx,xlsx_cells,7


,unit_id,format,source,locator,chars,assets,structured_kind
0,unit_9a806fdd6c116980865f,png,测试集A_产品架构图.png,"{""image"": ""测试集A_产品架构图.png"", ""width"": 1600, ""he...",66,1,NaN
1,unit_f6e122aa98e43ae28b3f,pdf,测试集A_产品规格与定价文档.pdf,"{""page"": 1, ""bbox"": [""0"", ""0"", ""595.2756"", ""84...",3910,0,pdf_page
2,unit_df05eff4da1b1fb4bbd6,pdf,测试集A_产品规格与定价文档.pdf,"{""page"": 2, ""bbox"": [""0"", ""0"", ""595.2756"", ""84...",5013,0,pdf_page
3,unit_471fd2a10f5966a52235,pdf,测试集A_产品规格与定价文档.pdf,"{""page"": 3, ""bbox"": [""0"", ""0"", ""595.2756"", ""84...",5013,0,pdf_page
4,unit_d66623961a03a057cdd5,pdf,测试集A_产品规格与定价文档.pdf,"{""page"": 3, ""table"": 1, ""bbox"": [""79.37008"", ""...",396,0,pdf_table
5,unit_ea0a0ab335e59e3d6ec9,pdf,测试集A_产品规格与定价文档.pdf,"{""page"": 4, ""bbox"": [""0"", ""0"", ""595.2756"", ""84...",5013,0,pdf_page
6,unit_be906a10248836f07cad,pdf,测试集A_产品规格与定价文档.pdf,"{""page"": 4, ""table"": 1, ""bbox"": [""79.37008"", ""...",1209,0,pdf_table
7,unit_c06e14b6ade5418ce3d5,pdf,测试集A_产品规格与定价文档.pdf,"{""page"": 5, ""bbox"": [""0"", ""0"", ""595.2756"", ""84...",5013,0,pdf_page
8,unit_3a5422d24c2b57d7a14a,pdf,测试集A_产品规格与定价文档.pdf,"{""page"": 5, ""table"": 1, ""bbox"": [""79.37008"", ""...",1290,0,pdf_table
9,unit_0e37b0b6d06df4cbd142,pdf,测试集A_产品规格与定价文档.pdf,"{""page"": 6, ""bbox"": [""0"", ""0"", ""595.2756"", ""84...",5013,0,pdf_page



--- 第一个 Unit 的 content_md 预览 ---
[Image: host must inspect original, labels, connectors and legend]

locator = {'image': '测试集A_产品架构图.png', 'width': 1600, 'height': 1100}
asset_refs = ['assets/doc_9e1b53c4d1c5f2e28e3c/original.png']


## Cell 8 — Prepare：看哪些东西根本不需要 LLM

Prepare 完全是本地 Python：

- 显式 alias table → deterministic；
- 普通 Excel 订单/指标 → deterministic；
- Excel 的备注/原因/风险 → semantic overlay；
- 真正复杂的自然语言 / 图片 → LLM。

### 如何理解结果

- `reviewed_without_llm` 越多，说明 Python 前置处理节省的模型调用越多；
- `pending_llm_units` 是还需要 OpenRouter 的 Unit 数；
- `mixed_semantic_units` 是“结构化主体由 Python 抽，只有自由文本交给模型”的 Excel Unit 数。

In [16]:
t0 = time.perf_counter()
prepare_result = run_cli('prepare', '--work', WORK_DIR)
prepare_elapsed = time.perf_counter() - t0

display(JSON(prepare_result))
print(f'Prepare wall time: {prepare_elapsed:.2f}s')

plan = json.loads((WORK_DIR / '00_extraction_plan.json').read_text(encoding='utf-8'))
plan_df = pd.DataFrame([{
    'unit_id': x['unit_id'],
    'mode': x.get('mode'),
    'priority': x.get('priority'),
    'reason': x.get('reason'),
} for x in plan['units']])
display(plan_df.groupby('mode').size().rename('count').reset_index())
display(plan_df.sort_values(['priority'], ascending=False).head(20))

<IPython.core.display.JSON object>

Prepare wall time: 0.53s


,mode,count
0,auto_no_facts,6
1,deterministic_alias_table,1
2,deterministic_xlsx,1
3,llm_batch,10
4,llm_semantic,17
5,llm_structured,15
6,llm_visual,1
7,xlsx_semantic_overlay,6


,unit_id,mode,priority,reason
0,unit_9a806fdd6c116980865f,llm_visual,103,None
55,unit_b5b5ea31a15a60447c61,xlsx_semantic_overlay,92,None
56,unit_3ed94ac917e3e83a86df,xlsx_semantic_overlay,92,None
54,unit_3285a456d4b70a1e71bf,xlsx_semantic_overlay,92,None
52,unit_eefbb2d6cbbcd6cfb974,xlsx_semantic_overlay,90,None
51,unit_b89a99447853446ed92a,xlsx_semantic_overlay,90,None
50,unit_474a4d247e1ede7bd691,xlsx_semantic_overlay,88,None
53,unit_b75d60928b41cf678219,deterministic_xlsx,88,None
30,unit_f8adb3b32a2347a4a87e,llm_structured,81,None
32,unit_b4c54b6155eb9f8d06d8,llm_structured,80,None


## Cell 9 — 输入 OpenRouter API Key，并定义并发模型调用器

API Key 用 `getpass()` 输入，不会显示，也不会写进 Notebook。

### 并发是怎么实现的？

Notebook 使用 `ThreadPoolExecutor`：

```text
Extraction wave
  batch 0 ┐
  batch 1 │
  batch 2 ├── 最多 8 个 HTTP 请求同时发给 OpenRouter
  ...     │
  batch 7 ┘
```

模型请求完成后，**accept 仍然顺序执行**，避免多个线程同时改 `state.json`。

Alignment 也是一样：同一 round 的模型判断并发，接受结果时顺序提交。

每次模型请求都会记录：

- stage / batch；
- task 数；
- wall time；
- prompt / completion / total tokens；
- HTTP / JSON 错误；
- 原始模型文本保存路径。

In [17]:
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests, base64, mimetypes, re, traceback

OPENROUTER_API_KEY = getpass('OpenRouter API Key: ').strip()
assert OPENROUTER_API_KEY, 'API Key 不能为空'

call_logs = []

EXTRACTION_SYSTEM = r"""你负责企业异构文档的语义抽取。速度和召回优先。只输出 JSON，不要解释。
对每个 task 只返回 semantic delta：
- mentions: local_id, type, name；只有确有别名/标识时才加 aliases/identifiers。
- relations: subject, predicate, object；可带必要 qualifiers。subject 和 object 必须是单个非空字符串，引用本 task 的 mentions.local_id 或输入 known_mentions.local_id；不得输出列表、字典、null 或数字，不得使用未声明实体的名称。
- 若一个实体与多个实体有关系，拆成多条 relations，每条只引用一个 object；需要的新实体先在 mentions 中声明。不要把多个实体 ID 放入一个 object。
- 文档格式列表、功能列表、数值及其他属性值放入 assertions，不要放入 relations.object；不要为了普通属性值创建实体。
- assertions: subject 必须为单个实体 local_id 字符串；普通属性使用 predicate=attribute、qualifiers.attribute=属性名、value=原始属性值。value 可以是字符串、数字、列表或 JSON 对象；可带必要 qualifiers。
- 例如格式列表应写为 {"subject":"d1","predicate":"attribute","qualifiers":{"attribute":"document_formats"},"value":["PDF","Word"]}，其中 d1 必须已在 mentions 或 known_mentions 中声明。
- 不要重复 known_mentions，不要重述 Python 已经抽出的结构字段。
- 默认不要输出 quote/evidence/review/reason；宿主会自动继承 Unit locator。
- unknown relation predicate 可以保留，不必强行改成白名单。
- unresolved 只写真正影响语义的歧义，不要写格式问题。
可用实体类型优先：Product, AITool, Actor, DataAsset, Version, Event, Observation。
返回一个对象：{"tasks":[...]}。每个输出 task 必须保留输入 task_id。不要输出 Markdown code fence。"""

ALIGNMENT_SYSTEM = r"""你负责实体 cluster 对齐。只做选择，不写分析过程。
对每个 task：比较 source_cluster 与 candidates；若同一稳定实体，decision 返回候选 entity_id；明显新实体返回 NEW_ENTITY；信息确实不足才返回 UNCERTAIN。
优先利用名称/别名/稳定 ID/已确认邻居；不要因为文档上下文不同就保守拆分同一稳定实体。
只能从该 task 给出的 candidate entity_id、NEW_ENTITY、UNCERTAIN 中选择。
返回：{"tasks":[{"task_id":"...","decision":"..."}, ...]}。不要输出 Markdown code fence。"""

def _extract_json_object(text):
    if not isinstance(text, str):
        raise ValueError('模型 content 不是字符串')
    s = text.strip()
    s = re.sub(r'^```(?:json)?\s*', '', s, flags=re.I)
    s = re.sub(r'\s*```$', '', s)
    try:
        return json.loads(s)
    except Exception:
        start, end = s.find('{'), s.rfind('}')
        if start >= 0 and end > start:
            return json.loads(s[start:end+1])
        raise

def _data_url(path: Path):
    mime = mimetypes.guess_type(path.name)[0] or 'image/png'
    data = base64.b64encode(path.read_bytes()).decode('ascii')
    return f'data:{mime};base64,{data}'

def _batch_content(stage, batch, fingerprint, round_no=None):
    envelope = {
        'stage': stage,
        'fingerprint': fingerprint,
        'tasks': batch['tasks'],
    }
    if round_no is not None:
        envelope['round'] = round_no
    text = '处理下面这一批任务，并严格保留每个 task_id：\n' + json.dumps(envelope, ensure_ascii=False, separators=(',', ':'))
    parts = [{'type': 'text', 'text': text}]

    # 视觉任务：把 asset_refs 直接作为多模态图片附加给 OpenRouter。
    seen_assets = set()
    for task in batch['tasks']:
        for ref in task.get('asset_refs', []) or []:
            asset = (WORK_DIR / ref).resolve()
            if not asset.exists() or asset in seen_assets:
                continue
            seen_assets.add(asset)
            parts.append({'type': 'text', 'text': f'下面图片属于 task_id={task["task_id"]}, asset_ref={ref}'})
            parts.append({'type': 'image_url', 'image_url': {'url': _data_url(asset)}})
    return parts

def call_openrouter_batch(stage, batch, fingerprint, round_no=None):
    assert stage in {'extract', 'align'}
    system = EXTRACTION_SYSTEM if stage == 'extract' else ALIGNMENT_SYSTEM
    reasoning = EXTRACT_REASONING if stage == 'extract' else ALIGN_REASONING
    max_tokens = EXTRACT_MAX_TOKENS if stage == 'extract' else ALIGN_MAX_TOKENS
    batch_index = batch.get('batch_index', 0)

    payload = {
        'model': MODEL,
        'messages': [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': _batch_content(stage, batch, fingerprint, round_no)},
        ],
        'max_tokens': max_tokens,
        'stream': False,
    }
    if reasoning:
        payload['reasoning'] = {'effort': reasoning}

    headers = {
        'Authorization': 'Bearer ' + OPENROUTER_API_KEY,
        'Content-Type': 'application/json',
    }

    last_exc = None
    for attempt in range(MAX_NETWORK_RETRIES + 1):
        t0 = time.perf_counter()
        try:
            r = requests.post(
                OPENROUTER_BASE_URL.rstrip('/') + '/chat/completions',
                headers=headers,
                json=payload,
                timeout=(30, HTTP_TIMEOUT),
            )
            elapsed = time.perf_counter() - t0
            r.raise_for_status()
            body = r.json()
            content = body.get('choices', [{}])[0].get('message', {}).get('content', '')
            obj = _extract_json_object(content)
            if not isinstance(obj, dict):
                raise ValueError('模型顶层结果不是 JSON object')
            if not isinstance(obj.get('tasks'), list) or not obj['tasks']:
                raise ValueError('模型结果缺少非空 tasks[]')

            # fingerprint / round 是 host 已知元数据，直接补，不浪费模型输出 token。
            obj['fingerprint'] = fingerprint
            if stage == 'align':
                obj['round'] = round_no

            usage = body.get('usage') or {}
            raw_txt = MODEL_OUTPUT_DIR / f'{stage}_batch_{batch_index:02d}_{int(time.time()*1000)}.txt'
            raw_json = raw_txt.with_suffix('.json')
            raw_txt.write_text(content if isinstance(content, str) else str(content), encoding='utf-8')
            raw_json.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')

            log = {
                'stage': stage,
                'batch_index': batch_index,
                'tasks': len(batch['tasks']),
                'attempt': attempt + 1,
                'status': 'ok',
                'elapsed_s': round(elapsed, 3),
                'prompt_tokens': usage.get('prompt_tokens'),
                'completion_tokens': usage.get('completion_tokens'),
                'total_tokens': usage.get('total_tokens'),
                'cost': usage.get('cost'),
                'raw_path': str(raw_txt),
            }
            call_logs.append(log)
            return {'response': obj, 'raw_content': content, 'body': body, 'log': log}
        except Exception as exc:
            elapsed = time.perf_counter() - t0
            last_exc = exc
            if attempt >= MAX_NETWORK_RETRIES:
                log = {
                    'stage': stage,
                    'batch_index': batch_index,
                    'tasks': len(batch['tasks']),
                    'attempt': attempt + 1,
                    'status': 'failed',
                    'elapsed_s': round(elapsed, 3),
                    'error': repr(exc),
                }
                call_logs.append(log)
                raise
            time.sleep(1.5 * (2 ** attempt))
    raise last_exc

def run_concurrent_batches(stage, task_export):
    batches = task_export.get('batches') or [
        {'batch_index': 0, 'tasks': task_export.get('tasks', [])}
    ]
    batches = [b for b in batches if b.get('tasks')]
    if not batches:
        return []
    concurrency = EXTRACT_CONCURRENCY if stage == 'extract' else ALIGN_CONCURRENCY
    round_no = task_export.get('round') if stage == 'align' else None
    fingerprint = task_export['fingerprint']

    results = []
    with ThreadPoolExecutor(max_workers=min(concurrency, len(batches))) as pool:
        futs = {
            pool.submit(call_openrouter_batch, stage, batch, fingerprint, round_no): batch
            for batch in batches
        }
        for fut in as_completed(futs):
            batch = futs[fut]
            try:
                result = fut.result()
                results.append((batch, result))
                print(f'[{stage}] batch={batch.get("batch_index")} tasks={len(batch["tasks"])} '
                      f'time={result["log"]["elapsed_s"]:.1f}s tokens={result["log"].get("total_tokens")}')
            except Exception as exc:
                print(f'[{stage}] batch={batch.get("batch_index")} FAILED: {exc!r}')
                raise
    results.sort(key=lambda x: x[0].get('batch_index', 0))
    return results

## Cell 10 — 先生成一波 Extraction 并发任务，但还不调用模型

这一格只调用 Skill 的：

```bash
kg_pipeline.py tasks --stage extract
```

并设置：

- 每个请求最多 5 个 task；
- 每个请求大约最多 14k semantic chars；
- 一次生成最多 8 个互不重叠 batch。

### 你应该看什么？

- `pending_count`：还有多少 Unit 尚未完成语义抽取；
- `batches`：这一波能并发的真实 HTTP 请求；
- 每个 batch 的 task 数 / chars；
- task 中的 `selection.mode`：是自然语言、Excel semantic overlay 还是视觉任务；
- `known_mentions`：Excel deterministic 部分已经抽过的实体，模型不要重复输出。

In [18]:
import pandas as pd

In [19]:
extract_task_file = RUN_ROOT / 'extract_batches_preview.json'
extract_export_meta = run_cli(
    'tasks', '--stage', 'extract', '--work', WORK_DIR,
    '--limit', EXTRACT_BATCH_SIZE,
    '--char-budget', EXTRACT_CHAR_BUDGET,
    '--batch-count', EXTRACT_CONCURRENCY,
    '--output', extract_task_file,
)
extract_export = json.loads(extract_task_file.read_text(encoding='utf-8'))

display(JSON(extract_export_meta))

batch_summary = []
for b in extract_export.get('batches', []):
    batch_summary.append({
        'batch_index': b['batch_index'],
        'tasks': len(b['tasks']),
        'batch_chars': b.get('batch_chars'),
        'modes': ', '.join(t.get('selection', {}).get('mode', '') for t in b['tasks']),
        'assets': sum(len(t.get('asset_refs', [])) for t in b['tasks']),
    })
display(pd.DataFrame(batch_summary))

if extract_export.get('batches'):
    first_task = extract_export['batches'][0]['tasks'][0]
    print('\n--- 第一条 extraction task（结构预览）---')
    display(JSON(first_task))

<IPython.core.display.JSON object>

,batch_index,tasks,batch_chars,modes,assets
0,0,1,66,llm_visual,1
1,1,5,11183,"xlsx_semantic_overlay, xlsx_semantic_overlay, ...",0
2,2,5,3343,"xlsx_semantic_overlay, llm_structured, llm_str...",0
3,3,5,4298,"llm_structured, llm_structured, llm_structured...",0
4,4,5,3999,"llm_structured, llm_structured, llm_structured...",0
5,5,5,6467,"llm_structured, llm_semantic, llm_semantic, ll...",0
6,6,4,13993,"llm_semantic, llm_semantic, llm_batch, llm_batch",0
7,7,5,2634,"llm_semantic, llm_semantic, llm_semantic, llm_...",0



--- 第一条 extraction task（结构预览）---


<IPython.core.display.JSON object>

## Cell 11 — 真正发起一波 8 路并发 Extraction，并查看模型原始输出

现在才真正访问 OpenRouter。

这一格会：

1. 同时发出最多 8 个 batch；
2. 等模型全部返回；
3. 在 Notebook 中显示每个 batch 的原始文本前 `RAW_PREVIEW_CHARS` 字符；
4. 完整原始输出写到 `kg_debug_run/model_outputs/`；
5. **此时还没有 accept**，方便你先观察模型到底输出了什么。

重点观察：

- 模型有没有仍然重复长 quote/evidence；
- 一个 task 是否只抽真正的 semantic delta；
- Excel semantic overlay 是否只处理备注/风险等自由文本；
- `completion_tokens` 是否明显比旧版小。

In [20]:
extract_wave_results = run_concurrent_batches('extract', extract_export)

for batch, result in extract_wave_results:
    print('\n' + '=' * 100)
    print(f'EXTRACT BATCH {batch["batch_index"]} | {len(batch["tasks"])} tasks | {result["log"]}')
    raw = result['raw_content'] or ''
    print(raw[:RAW_PREVIEW_CHARS])
    if len(raw) > RAW_PREVIEW_CHARS:
        print(f'\n... [Notebook 仅预览前 {RAW_PREVIEW_CHARS} chars；完整结果见 {result["log"]["raw_path"]}]')

[extract] batch=6 tasks=4 time=16.9s tokens=5464
[extract] batch=7 tasks=5 time=23.6s tokens=7346
[extract] batch=4 tasks=5 time=30.4s tokens=9177
[extract] batch=1 tasks=5 time=45.4s tokens=26734
[extract] batch=2 tasks=5 time=46.5s tokens=12364
[extract] batch=3 tasks=5 time=51.7s tokens=12537
[extract] batch=0 tasks=1 time=76.0s tokens=11128
[extract] batch=5 tasks=5 time=110.3s tokens=15536

EXTRACT BATCH 0 | 1 tasks | {'stage': 'extract', 'batch_index': 0, 'tasks': 1, 'attempt': 1, 'status': 'ok', 'elapsed_s': 76.034, 'prompt_tokens': 3272, 'completion_tokens': 7856, 'total_tokens': 11128, 'cost': 0, 'raw_path': 'C:\\Users\\Agesen\\Downloads\\Task4\\github-release\\evidence-kg-graphrag\\kg_debug_run_q5q10_fix\\model_outputs\\extract_batch_00_1791183559822.txt'}
{"tasks":[{"task_id":"unit_9a806fdd6c116980865f","mentions":[{"local_id":"d1","type":"Product","name":"明创工坊产品架构图","identifiers":["v1.0"]},{"local_id":"org1","type":"Actor","name":"明创工坊","aliases":["MingCraft Studio"]},{"loc

## Cell 12 — 接受这一波 Extraction 结果

模型请求可以并发，但对 `state.json` 的写入不要并发。

所以这里按 batch 顺序执行 `accept`。

当前 speed-first Skill 的 acceptor 很宽松：

- 不要求逐字 quote；
- endpoint 写唯一名称/alias 也会自动 resolve；
- unknown relation predicate 会保留；
- 普通 JSON number 直接接受；
- 一个 item 有问题只跳过那个 item，不会把整 Unit quarantine。

你会看到每次 accept 后还剩多少 `pending_extraction`。

In [21]:
accept_reports = []
for batch, result in extract_wave_results:
    response_path = MODEL_OUTPUT_DIR / f'accepted_extract_batch_{batch["batch_index"]:02d}_{int(time.time()*1000)}.json'
    response_path.write_text(json.dumps(result['response'], ensure_ascii=False, indent=2), encoding='utf-8')
    report = run_cli('accept', '--stage', 'extract', '--work', WORK_DIR, '--response', response_path)
    accept_reports.append(report)
    print(f'batch {batch["batch_index"]}: accepted={report.get("accepted")} '
          f'pending={report.get("pending_extraction")} reviewed={report.get("reviewed_units")}')

display(pd.DataFrame(accept_reports))

batch 0: accepted=1 pending=48 reviewed=9
batch 1: accepted=5 pending=43 reviewed=14
batch 2: accepted=5 pending=38 reviewed=19
batch 3: accepted=5 pending=33 reviewed=24
batch 4: accepted=5 pending=28 reviewed=29
batch 5: accepted=5 pending=23 reviewed=34
batch 6: accepted=4 pending=19 reviewed=38
batch 7: accepted=5 pending=14 reviewed=43


,ok,stage,accepted,reviewed_units,pending_extraction,round
0,True,extract,1,9,48,1
1,True,extract,5,14,43,1
2,True,extract,5,19,38,1
3,True,extract,5,24,33,1
4,True,extract,5,29,28,1
5,True,extract,5,34,23,1
6,True,extract,4,38,19,1
7,True,extract,5,43,14,1


## Cell 13 — 自动跑完剩余 Extraction（仍然 8 路并发）

上一格只跑了一波，是为了让你手工看清模型行为。

这一格会循环：

```text
生成最多 8 个不重叠 batch
        ↓
8 路 OpenRouter 并发
        ↓
顺序 accept
        ↓
直到 pending_extraction = 0
```

每一波仍会把完整原始结果保存到 `model_outputs/`，但 Notebook 只打印摘要，避免页面被几十万字符刷爆。

In [22]:
def finish_extraction():
    wave = 0
    while True:
        wave += 1
        task_file = RUN_ROOT / f'extract_wave_{wave:02d}.json'
        meta = run_cli(
            'tasks', '--stage', 'extract', '--work', WORK_DIR,
            '--limit', EXTRACT_BATCH_SIZE,
            '--char-budget', EXTRACT_CHAR_BUDGET,
            '--batch-count', EXTRACT_CONCURRENCY,
            '--output', task_file,
        )
        export = json.loads(task_file.read_text(encoding='utf-8'))
        batches = export.get('batches', [])
        if not batches:
            print('Extraction 已全部完成。')
            break
        print(f'\n[Extraction wave {wave}] pending={export.get("pending_count")} batches={len(batches)} '
              f'tasks={sum(len(b["tasks"]) for b in batches)}')
        results = run_concurrent_batches('extract', export)
        for batch, result in results:
            path = MODEL_OUTPUT_DIR / f'extract_wave{wave:02d}_batch{batch["batch_index"]:02d}_accepted.json'
            path.write_text(json.dumps(result['response'], ensure_ascii=False, indent=2), encoding='utf-8')
            rep = run_cli('accept', '--stage', 'extract', '--work', WORK_DIR, '--response', path)
            print(f'  accept batch={batch["batch_index"]} pending={rep.get("pending_extraction")}')

finish_extraction()


[Extraction wave 1] pending=14 batches=6 tasks=14
[extract] batch=2 tasks=2 time=12.8s tokens=3785
[extract] batch=0 tasks=4 time=18.8s tokens=7796
[extract] batch=5 tasks=2 time=19.1s tokens=4723
[extract] batch=4 tasks=2 time=20.1s tokens=4592
[extract] batch=3 tasks=2 time=31.4s tokens=4910
[extract] batch=1 tasks=2 time=31.9s tokens=7790
  accept batch=0 pending=10
  accept batch=1 pending=8
  accept batch=2 pending=6
  accept batch=3 pending=4
  accept batch=4 pending=2
  accept batch=5 pending=0
Extraction 已全部完成。


## Cell 14 — 检查 Extraction 产物

这里不是做“审查”，而是为了让你肉眼看看模型到底抽成了什么。

重点文件：

- `01_mentions.json`：mention 层实体，尚未全部 canonicalize；
- `state.json`：当前 workflow checkpoint；
- 每个 extraction item 的 `warnings`：只表示 host 做过跳过/修复，不会阻止构图。

你可以看：

- mention 数量和类型分布；
- relation/assertion 总量；
- warning 数量；
- 是否还有大量人工制造的 unresolved（正常情况下应该很少）。

In [23]:
state = json.loads((WORK_DIR / 'state.json').read_text(encoding='utf-8'))
extractions = list(state.get('extractions', {}).values())

mentions = [m for ex in extractions for m in ex.get('mentions', [])]
relations = [r for ex in extractions for r in ex.get('relations', [])]
assertions = [a for ex in extractions for a in ex.get('assertions', [])]
unresolved = [u for ex in extractions for u in ex.get('unresolved', [])]
warnings = [w for ex in extractions for w in ex.get('warnings', [])]

print('extraction units =', len(extractions))
print('mentions         =', len(mentions))
print('relations        =', len(relations))
print('assertions       =', len(assertions))
print('model unresolved =', len(unresolved))
print('host warnings    =', len(warnings))

display(pd.DataFrame(mentions).groupby('type').size().rename('count').reset_index().sort_values('count', ascending=False))
print('\n随机/前几条 relation：')
display(pd.DataFrame(relations).head(15))
print('\n前几条 assertion：')
display(pd.DataFrame(assertions).head(15))

extraction units = 57
mentions         = 932
relations        = 564
assertions       = 801
model unresolved = 0
host warnings    = 140


,type,count
5,Product,431
1,Actor,187
3,Event,127
0,AITool,114
6,Version,52
4,Observation,14
2,DataAsset,7



随机/前几条 relation：


,subject,predicate,object,qualifiers,evidence,origin
0,m_451ddbda5f42eb0aea54,purchased,m_2a3e556bfe2c90bbeeed,"{'record_id': 'ORD-20260713-0115', 'time': '20...","[{'unit_id': 'unit_b75d60928b41cf678219', 'doc...",asserted
1,m_c87300f873309fb8ba9c,purchased,m_e4f566a3b21995068322,"{'record_id': 'ORD-20260715-0116', 'time': '20...","[{'unit_id': 'unit_b75d60928b41cf678219', 'doc...",asserted
2,m_1fbea6b648b70a30ef9a,purchased,m_3414c8e4aa6e931563a8,"{'record_id': 'ORD-20260718-0117', 'time': '20...","[{'unit_id': 'unit_b75d60928b41cf678219', 'doc...",asserted
3,m_92255bf0360791d52f61,purchased,m_6f4f1b6b3acc36d388d0,"{'record_id': 'ORD-20260720-0118', 'time': '20...","[{'unit_id': 'unit_b75d60928b41cf678219', 'doc...",asserted
4,m_972295cef27e5f9eb7dd,purchased,m_b036e2d78648d73cb51a,"{'record_id': 'ORD-20260723-0119', 'time': '20...","[{'unit_id': 'unit_b75d60928b41cf678219', 'doc...",asserted
5,m_5e0a1e283a5e9bf7e345,purchased,m_0a0bcded12e744337ea5,"{'record_id': 'ORD-20260725-0120', 'time': '20...","[{'unit_id': 'unit_b75d60928b41cf678219', 'doc...",asserted
6,m_f1c8d6ab2425341fd03b,维护方,m_36057fd76c9772808330,{},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted
7,m_8b04f6cf0c9a50c9fe97,基于开发,m_4bf76fa4d7a8b3a58939,{},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted
8,m_429455e834963b3664e5,已并入,m_7310cfc19ee7c4096151,{},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted
9,m_019a971ed816e8b05430,替代,m_3e0d74890685ebebe263,{},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted



前几条 assertion：


,subject,predicate,value,qualifiers,evidence,origin
0,m_f1c8d6ab2425341fd03b,attribute,v1.0,{'attribute': 'version'},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted
1,m_f1c8d6ab2425341fd03b,attribute,2026-06-30,{'attribute': 'update_date'},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted
2,m_f1c8d6ab2425341fd03b,attribute,张明·明创工坊(MingCraft Studio),{'attribute': 'editor'},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted
3,m_f1c8d6ab2425341fd03b,attribute,4,{'attribute': 'free_product_count'},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted
4,m_f1c8d6ab2425341fd03b,attribute,14,{'attribute': 'paid_product_count'},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted
5,m_f1c8d6ab2425341fd03b,attribute,18,{'attribute': 'ai_tool_count'},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted
6,m_f1c8d6ab2425341fd03b,attribute,15,{'attribute': 'customer_persona_count'},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted
7,m_8b04f6cf0c9a50c9fe97,attribute,核心产品,{'attribute': 'product_role'},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted
8,m_8b04f6cf0c9a50c9fe97,attribute,"{'raw': '', 'kind': 'text', 'value': ''}",{'attribute': 'document_formats'},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted
9,m_3e0d74890685ebebe263,attribute,停售,{'attribute': 'status'},"[{'unit_id': 'unit_9a806fdd6c116980865f', 'doc...",asserted


## Cell 15 — Alignment：按 round 并发实体对齐

Alignment 的并发方式和 Extraction 不同：

```text
Round 1 frozen snapshot
   ├─ batch 0 ┐
   ├─ batch 1 │  8 路并发模型判断
   ├─ ...     │
   └─ batch 7 ┘
          ↓
      顺序 accept
          ↓
如果发生 cluster merge → Round 2
否则停止
```

模型每个 task 只应该返回：

```json
{"task_id":"align_...","decision":"entity_..."}
```

或 `NEW_ENTITY / UNCERTAIN`。

**不需要 reason，不需要 evidence_ids。**

下面会自动跑完所有 round，同时把每轮模型原始输出保存下来。

In [24]:
def finish_alignment():
    wave = 0
    while True:
        wave += 1
        task_file = RUN_ROOT / f'align_wave_{wave:02d}.json'
        meta = run_cli(
            'tasks', '--stage', 'align', '--work', WORK_DIR,
            '--limit', ALIGN_BATCH_SIZE,
            '--batch-count', ALIGN_CONCURRENCY,
            '--output', task_file,
        )
        export = json.loads(task_file.read_text(encoding='utf-8'))
        batches = export.get('batches', [])
        print(f'\n[Alignment wave {wave}] round={export.get("round")} pending={export.get("pending_count")} '
              f'batches={len(batches)} stopped={export.get("stopped")}')
        if not batches:
            if export.get('stopped'):
                print('Alignment 已停止。')
                break
            # 理论上若没有 task，Skill 会自动 stop；重新取一次状态。
            continue

        results = run_concurrent_batches('align', export)
        for batch, result in results:
            print(f'  raw decisions batch={batch["batch_index"]}: {str(result["raw_content"])[:1200]}')
            path = MODEL_OUTPUT_DIR / f'align_wave{wave:02d}_batch{batch["batch_index"]:02d}_accepted.json'
            path.write_text(json.dumps(result['response'], ensure_ascii=False, indent=2), encoding='utf-8')
            rep = run_cli('accept', '--stage', 'align', '--work', WORK_DIR, '--response', path)
            print(f'  accept batch={batch["batch_index"]}: round_now={rep.get("round")}')

finish_alignment()


[Alignment wave 1] round=1 pending=97 batches=8 stopped=False
[align] batch=6 tasks=8 time=5.8s tokens=8793
[align] batch=3 tasks=8 time=6.5s tokens=10107
[align] batch=7 tasks=8 time=7.0s tokens=12334
[align] batch=0 tasks=8 time=7.2s tokens=7349
[align] batch=1 tasks=8 time=7.6s tokens=10711
[align] batch=5 tasks=8 time=8.2s tokens=12475
[align] batch=2 tasks=8 time=8.2s tokens=7383
[align] batch=4 tasks=8 time=11.5s tokens=10431
  raw decisions batch=0: {"tasks":[{"task_id":"align_e67ec6a4beb754adf793","decision":"NEW_ENTITY"},{"task_id":"align_3bc9287eeaa8df6dab9f","decision":"NEW_ENTITY"},{"task_id":"align_e26cf9775219ac9b4a74","decision":"NEW_ENTITY"},{"task_id":"align_83f5c20fdc2a04810139","decision":"NEW_ENTITY"},{"task_id":"align_9a65294ae4efc9d6f0a6","decision":"entity_eecf347aa4a12d7bd9c3"},{"task_id":"align_da891179ca98786435ad","decision":"NEW_ENTITY"},{"task_id":"align_477161be99c2da377c1e","decision":"NEW_ENTITY"},{"task_id":"align_2862ceed67594d8f55a4","decision":"NEW_

## Cell 16 — 看 Alignment 到底做成什么样

这格主要看 `state.json` 中的：

- `alignment.mapping`：mention → canonical cluster；
- `alignment.clusters`：每个 canonical entity 聚了多少 mention；
- `cluster_status`：resolved / uncertain 等；
- `history`：deterministic merge 和模型 cluster merge 的历史。

重点不是“uncertain 必须为 0”，而是看：

1. 同一稳定产品/工具有没有仍被切成大量碎片；
2. 明显不同版本有没有被错误合并；
3. LLM alignment task 数是否已经比原 mention 数小很多。

In [25]:
state = json.loads((WORK_DIR / 'state.json').read_text(encoding='utf-8'))
al = state['alignment']

print('alignment stopped =', al.get('stopped'))
print('round             =', al.get('round'))
print('clusters          =', len(al.get('clusters', {})))
print('mapping mentions  =', len(al.get('mapping', {})))
print('history entries   =', len(al.get('history', [])))
print('status counts     =', Counter(al.get('cluster_status', {}).values()))

cluster_sizes = sorted(
    [(eid, len(members), al.get('cluster_status', {}).get(eid)) for eid, members in al.get('clusters', {}).items()],
    key=lambda x: -x[1]
)
display(pd.DataFrame(cluster_sizes[:30], columns=['entity_id', 'member_count', 'status']))

print('\n最近 20 条 alignment history：')
display(pd.DataFrame(al.get('history', [])[-20:]))

alignment stopped = True
round             = 2
clusters          = 283
mapping mentions  = 932
history entries   = 707
status counts     = Counter({'record': 141, 'resolved': 139, 'uncertain': 3})


,entity_id,member_count,status
0,entity_d5f26d252a1cc6aa187c,109,resolved
1,entity_fd0fe0ba7cd2e4409e98,106,resolved
2,entity_fcd75a56ca9937bb026d,35,resolved
3,entity_1d449f99af77ca73a3f6,31,resolved
4,entity_d4898a2f307424a704d0,21,resolved
5,entity_8018dafe2ab70564238e,19,resolved
6,entity_b21a3942ea1cbadb1eb7,18,resolved
7,entity_f1ee559b9068c599e8dc,18,resolved
8,entity_68367bb1fead3154e08d,16,resolved
9,entity_53e71ac20de466a80d50,14,resolved



最近 20 条 alignment history：


,round,decision,entity_id,source_entity_id,target_entity_id,source_members,target_members
0,1,NEW_ENTITY,entity_c56b96cd30b4963c489f,NaN,NaN,NaN,NaN
1,1,cluster_merge,entity_df77fbb6b3872ae31ba2,entity_c5e68f4da2f99cfd7a5a,entity_df77fbb6b3872ae31ba2,"[m_66ad0fbb99452f9bc479, m_8c565672776ae22e02c8]","[m_37efd2ac93c5591eaa78, m_9616caece628900b232..."
2,1,NEW_ENTITY,entity_cad03485361b40e14cd0,NaN,NaN,NaN,NaN
3,1,NEW_ENTITY,entity_cad74cbe5713409c6e5c,NaN,NaN,NaN,NaN
4,1,NEW_ENTITY,entity_cb41f9c089ece0f359da,NaN,NaN,NaN,NaN
5,1,cluster_merge,entity_fcd75a56ca9937bb026d,entity_ced101c22b4e0453d8c4,entity_fcd75a56ca9937bb026d,[m_3e498ffe282ca6720502],"[m_1545ba1282e3a6f5ece9, m_1c45cc4a627524815ac..."
6,1,NEW_ENTITY,entity_d5f26d252a1cc6aa187c,NaN,NaN,NaN,NaN
7,1,cluster_merge,entity_fb25eb365b2f1d5a38ab,entity_da6358019f7a7c65d175,entity_fb25eb365b2f1d5a38ab,"[m_5284a94c47d97f61d98c, m_7769da07dfd1d226965...","[m_26bc465b2735bf83fab5, m_57c7049ed2e55b36f3b..."
8,1,NEW_ENTITY,entity_dea783661517d69d495d,NaN,NaN,NaN,NaN
9,1,UNCERTAIN,entity_df013551462a0b4e49f3,NaN,NaN,NaN,NaN


## Cell 17 — Build：生成 Canonical Facts 和最终稀疏 KG

Build 阶段不调用 LLM。

它会生成：

- `02_entity_map.json`：mention → canonical entity；
- `03_canonical_facts.json`：事实层 + evidence；
- `04_knowledge_graph.json`：稀疏 traversal graph。

当前 speed-first 版本只做**最小引用完整性检查**，不会再因为 quote、raw literal、domain/range 或 unresolved 数量阻塞 Build。

In [26]:
t0 = time.perf_counter()
build_result = run_cli('build', '--work', WORK_DIR)
build_elapsed = time.perf_counter() - t0

display(JSON(build_result))
print(f'Build wall time: {build_elapsed:.2f}s')

validate_result = run_cli('validate', '--work', WORK_DIR, '--require-complete', check=False)
print('\n--- 最小完整性 validate ---')
display(JSON(validate_result))

<IPython.core.display.JSON object>

Build wall time: 0.94s

--- 最小完整性 validate ---


<IPython.core.display.JSON object>

## Cell 18 — 看最终 KG 的结构和规模

这格用于判断“构图是否太稀/太碎/太肥”。

主要看：

- `nodes`：稳定实体 / Event / Version 等；
- `facts`：绝大多数属性、指标、日期应该留在这里；
- `edges`：真正适合 traversal 的稀疏关系；
- `evidence`：来源 locator；
- `diagnostics`：流程是否完整。

一个健康的 Fact-first KG 往往应该是：**facts 明显多于 nodes/edges**，而不是把每个数字都建成节点。

In [27]:
graph = json.loads((WORK_DIR / '04_knowledge_graph.json').read_text(encoding='utf-8'))
print('nodes    =', len(graph.get('nodes', [])))
print('edges    =', len(graph.get('edges', [])))
print('facts    =', len(graph.get('facts', [])))
print('evidence =', len(graph.get('evidence', [])))
print('unresolved diagnostics =', len(graph.get('diagnostics', {}).get('unresolved', [])))

node_types = Counter(n.get('type') for n in graph.get('nodes', []))
edge_types = Counter(e.get('predicate') for e in graph.get('edges', []))
print('\nNode types:', node_types)
print('\nTop edge predicates:', edge_types.most_common(20))

nodes    = 283
edges    = 424
facts    = 1334
evidence = 511
unresolved diagnostics = 0

Node types: Counter({'Event': 127, 'Product': 51, 'Actor': 38, 'AITool': 34, 'Version': 15, 'Observation': 14, 'DataAsset': 4})

Top edge predicates: [('affects', 40), ('has_participant', 30), ('responsible_actor', 22), ('has_update', 21), ('purchased', 18), ('responsible_person', 16), ('update_for', 16), ('uses', 16), ('version_of', 15), ('depends_on', 13), ('reviewed_by', 10), ('concerns', 6), ('observed_for', 6), ('使用', 5), ('replaced_by', 4), ('add_on_for', 4), ('updates', 4), ('supports', 4), ('merged_into', 3), ('triggers', 3)]


## Cell 19 — 可选：按关键词检查某个 canonical entity 的 aliases / alias_bindings

这格对排查跨格式名称题尤其有用。

例如你怀疑“明明 Alignment 已经知道五个别名，为什么回答里没出现”，就把 `ENTITY_SEARCH` 改成其中任意一个名字。

它会直接显示最终 node 的：

- canonical `name`；
- `aliases`；
- `alias_bindings`；
- `attribute_claims`。

如果这里就没有别名，问题在 Extraction/Alignment；如果这里有，但最终回答没有，问题就在 GraphRAG retrieval/context。

In [39]:
ENTITY_SEARCH = '张明'  # 例如填一个产品名、别名或工具名

if ENTITY_SEARCH.strip():
    q = ENTITY_SEARCH.casefold()
    hits = []
    for n in graph.get('nodes', []):
        labels = [n.get('name', ''), *n.get('aliases', [])]
        if any(q in str(x).casefold() for x in labels):
            hits.append(n)
    print(f'命中 {len(hits)} 个 node')
    for n in hits[:10]:
        display(JSON(n))
else:
    print('ENTITY_SEARCH 为空；需要时填入一个名称后重跑本格。')

命中 5 个 node


<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

## Cell 20 — Lean GraphRAG：只做召回，不调用最终回答模型

这是诊断召回效果最重要的一格。

把 `QUESTION` 改成你要测试的题目。运行后会分别显示：

1. **Query Plan**：Planner 把题目判断成什么类型；
2. **Entity Profiles**：实体名称、aliases、alias_bindings 是否进来了；
3. **Structured Hits**：Sheet/列/单元格/结构化来源；
4. **Fact Hits**：Canonical Fact BM25；
5. **Graph Paths**：只有真正适合多跳图推理的题才应该出现；
6. **Raw Units**：最终回原文补证据的来源；
7. **Context**：真正会送给最终 LLM 的文本。

如果“retrieval 已经找到，但 context 没有”，就是 context budget/组装问题；如果 retrieval 本身没有，就是 Planner 或检索通道问题。
本格可以在新内核中直接运行，无需先执行构图 cell；会加载完整配置和本地依赖，自动读取最新完整图谱，找不到时回退原 native 图谱。


In [40]:
# 独立准备本 cell 所需环境；不运行安装、解析、抽取、对齐或构图。
from pathlib import Path
from IPython.display import display, JSON, Markdown
import importlib, sys

_query_notebook = Path.cwd() / 'evidence_kg_openrouter_debug.ipynb'
if not _query_notebook.is_file():
    raise FileNotFoundError('请在本 notebook 所在目录启动内核。')
if str(_query_notebook.parent) not in sys.path:
    sys.path.insert(0, str(_query_notebook.parent))
import interactive_kg as _query_module
importlib.reload(_query_module)
_retrieval_runtime = _query_module.query_runtime(_query_notebook, overrides=globals())
call_logs = _retrieval_runtime['call_logs']
RAG_WORK_DIR = _retrieval_runtime['RAG_WORK_DIR']
_selected = _retrieval_runtime['RAG_GRAPH_SELECTION']
print(f'[图谱] source={_selected["source"]} backend={_selected["backend"]} '
      f'nodes={_selected["nodes"]} facts={_selected["facts"]} path={RAG_WORK_DIR}')

import time
import pandas as pd

# 把这里换成你要测试的问题。
QUESTION = """Excel"订单明细"Sheet的备注列中隐藏了6条关键线索（分布在不同的行中）。请找出全部6条隐藏线索，并按时间顺序串联成一条完整的事件链，说明这些事件之间的因果关系。""".strip()
assert QUESTION and '请在这里' not in QUESTION, '请先把 QUESTION 改成真实问题'

LeanGraphRAG = _retrieval_runtime['LeanGraphRAG']
ANSWER_PROMPT = _retrieval_runtime['ANSWER_PROMPT']
rag = LeanGraphRAG(
    RAG_WORK_DIR,
    max_hops=3,
    beam_width=20,
    context_chars=18_000,
)

t0 = time.perf_counter()
retrieval = rag.retrieve(QUESTION)
retrieval_elapsed = time.perf_counter() - t0

print(f'Retrieval wall time: {retrieval_elapsed:.3f}s')
print('\n=== Query Plan ===')
display(JSON(retrieval['query_plan']))

print('\n=== Stats ===')
display(JSON(retrieval['stats']))

print('\n=== Entity Profiles ===')
for x in retrieval.get('entity_profiles', []):
    display(JSON(x))

print('\n=== Structured Hits ===')
structured_rows = [{
    'score': h.get('score'), 'source': h.get('source'), 'locator': str(h.get('locator')),
    'field': h.get('field'), 'value': str(h.get('value'))[:300]
} for h in retrieval.get('structured_hits', [])]
display(pd.DataFrame(structured_rows).head(30))

print('\n=== Fact Hits ===')
fact_rows = []
for h in retrieval.get('fact_hits', []):
    f = h['fact']
    fact_rows.append({
        'score': h.get('score'), 'rrf': h.get('rrf_score'), 'fact_id': h.get('fact_id'),
        'subject': rag.nodes.get(f.get('subject'), {}).get('name'),
        'predicate': f.get('predicate'),
        'object': rag.nodes.get(f.get('object'), {}).get('name') if f.get('object') else None,
        'value': str(f.get('value'))[:220], 'qualifiers': str(f.get('qualifiers'))[:220],
    })
display(pd.DataFrame(fact_rows).head(30))

print('\n=== Graph Paths ===')
for i, p in enumerate(retrieval.get('paths', []), 1):
    chain = []
    for s in p.get('steps', []):
        chain.append(f"{rag.nodes.get(s.get('source'),{}).get('name',s.get('source'))} --{s.get('predicate')}--> {rag.nodes.get(s.get('target'),{}).get('name',s.get('target'))}")
    print(f'Path {i} score={p.get("score"):.3f}: ' + ' | '.join(chain))

print('\n=== Raw Units ===')
raw_rows = [{
    'score': x.get('score'), 'format': x.get('format'), 'source': x.get('source'),
    'locator': str(x.get('locator')), 'preview': x.get('content_md', '')[:500].replace('\n', ' ')
} for x in retrieval.get('raw_units', [])]
display(pd.DataFrame(raw_rows))

print('\n=== 真正送给最终模型的 Context ===')
print('context chars =', len(retrieval['context']))
print(retrieval['context'][:_retrieval_runtime['CONTEXT_PREVIEW_CHARS']])
if len(retrieval['context']) > _retrieval_runtime['CONTEXT_PREVIEW_CHARS']:
    print(f'\n... [仅预览前 {_retrieval_runtime["CONTEXT_PREVIEW_CHARS"]} chars]')

[图谱] source=latest backend=native nodes=286 facts=1344 path=C:\Users\Agesen\Downloads\Task4\github-release\evidence-kg-graphrag\kg_debug_run_q5q10_fix\work
Retrieval wall time: 1.023s

=== Query Plan ===


<IPython.core.display.JSON object>


=== Stats ===


<IPython.core.display.JSON object>


=== Entity Profiles ===


...


=== Fact Hits ===


,score,rrf,fact_id,subject,predicate,object,value,qualifiers
0,163.683257,0.044412,fact_33739b296cc10ddf8d2d,新客户注册量首次突破日均50人,observed_for,外贸公司市场部,None,{}
1,162.157183,0.044324,fact_06355595b43d5f5c6088,新客户注册量首次突破日均50人,concerns,API接口测试工具,None,{}
2,189.434857,0.016393,fact_755607ab6647a224e724,新客户注册量首次突破日均50人,attribute,NaN,50,{'attribute': '新客户注册量首次突破日均人数'}
3,126.488693,0.015625,fact_d861786777b012a8c4f0,全局定价策略变更,attribute,NaN,免费增值 → 订阅制,{'attribute': 'pricing_transition'}
4,115.394239,0.015385,fact_bc538aa32c7a8b0921db,极简笔记定价策略变更,attribute,NaN,免费增值→订阅制,{'attribute': '定价模式变更'}
5,98.351265,0.015152,fact_155fe16f97967386e8a9,code-assist-pro,收入受升级影响,工具升级兼容性影响,None,{'Q2收入下降贡献': '-12%'}
6,95.816799,0.014925,fact_17c7f2ca9ab275181ce8,定价策略变更评审,reviews,定价策略变更,None,{}
7,77.172767,0.014706,fact_7bf027ed4d6bb296b159,文案助手功能并入周报工具,attribute,NaN,2026-05-20,{'attribute': 'date'}
8,76.145750,0.014493,fact_6734b611cf6925137de6,全局定价策略变更,attribute,NaN,2026-04-08,{'attribute': 'date'}
9,75.185773,0.014286,fact_cf81e0e49b29f021a54c,定价策略变更方案评审,attribute,NaN,定价策略变更方案评审,{'attribute': '评审主题'}



=== Graph Paths ===
Path 1 score=24.691: 新客户注册量首次突破日均50人 --concerns--> API接口测试工具 | 高校计算机社团 --purchased--> API接口测试工具 | 高校计算机社团 --purchased--> code-assist-pro
Path 2 score=24.684: 新客户注册量首次突破日均50人 --observed_for--> 外贸公司市场部 | 外贸公司市场部 --purchased--> API接口测试工具 | API接口测试工具:版本更新 --affects--> API接口测试工具
Path 3 score=24.434: 新客户注册量首次突破日均50人 --concerns--> API接口测试工具 | API接口测试工具 --uses--> Cursor | code-assist-pro --uses--> Cursor
Path 4 score=24.138: 新客户注册量首次突破日均50人 --observed_for--> 外贸公司市场部 | 外贸公司市场部 --purchased--> API接口测试工具 | 高校计算机社团 --purchased--> API接口测试工具
Path 5 score=23.881: 新客户注册量首次突破日均50人 --observed_for--> 外贸公司市场部 | 外贸公司市场部 --purchased--> API接口测试工具 | API接口测试工具 --uses--> Cursor
Path 6 score=23.701: 新客户注册量首次突破日均50人 --concerns--> API接口测试工具 | API接口测试工具 --uses--> Cursor | 简历工具 --uses--> Cursor

=== Raw Units ===


,score,format,source,locator,preview
0,0.043947,excel,测试集A_运营数据记录.xlsx,"{'sheet': '订单明细', 'sheet_state': 'visible', 'r...",# Sheet: 订单明细 ## Original header candidates (...
1,0.040499,excel,测试集A_运营数据记录.xlsx,"{'sheet': '产品变更记录', 'sheet_state': 'visible', ...",# Sheet: 产品变更记录 ## Original header candidates...
2,0.038544,excel,测试集A_运营数据记录.xlsx,"{'sheet': 'AI工具使用统计', 'sheet_state': 'visible'...",# Sheet: AI工具使用统计 ## Original header candidat...
3,0.036883,excel,测试集A_运营数据记录.xlsx,"{'sheet': '收入与续费率统计', 'sheet_state': 'visible'...",# Sheet: 收入与续费率统计 ## Original header candidat...
4,0.045850,ppt,测试集A_产品运营分析报告.pptx,{'slide': 2},工作室概况 明创工坊（MingCraft Studio） 创立时间：2025年10月 运...
5,0.045527,word,测试集A_工作室运营手册.docx,"{'section': ['2. 产品开发与定价策略', '2.1 定价策略与AB实验'],...",## 2.1 定价策略与AB实验 2026年4月8日，工作室对核心产品code-assis...
6,0.043846,ppt,测试集A_产品运营分析报告.pptx,{'slide': 6},Cursor v2升级影响分析 事件时间线：2026-05-06 Cursor v1→v2...
7,0.043174,excel,测试集A_运营数据记录.xlsx,"{'sheet': '订单明细', 'sheet_state': 'visible', 'r...",# Sheet: 订单明细 ## Original header candidates (...
8,0.042776,pdf,测试集A_产品规格与定价文档.pdf,"{'page': 16, 'bbox': ['0', '0', '595.2756', '8...",明创工坊 · 产品规格与定价文档 ...
9,0.042732,excel,测试集A_运营数据记录.xlsx,"{'sheet': '订单明细', 'sheet_state': 'visible', 'r...",# Sheet: 订单明细 ## Original header candidates (...



=== 真正送给最终模型的 Context ===
context chars = 17829
QUESTION
Excel"订单明细"Sheet的备注列中隐藏了6条关键线索（分布在不同的行中）。请找出全部6条隐藏线索，并按时间顺序串联成一条完整的事件链，说明这些事件之间的因果关系。

QUERY_PLAN
{"intent":"structured_enumeration","structured":true,"cross_format_identity":false,"table_reconstruction":false,"retrieval_mode":"exhaustive","requested_formats":["excel"],"list_all":true,"causal":true,"comparison":false,"rank":false,"timeline":true,"expected_count":6,"sheet_hints":["订单明细"],"column_hints":["备注"],"years":[],"months":[],"quarters":[],"keywords":["excel","Excel\"订单明细\"","备注列","隐藏","6条关键线索","分布","不同","6条隐藏线索","并按时间顺序串联成一条完整","事件链","这些事件之间","关系"]}

ENTITY_PROFILE {"id":"entity_7ff20452aaca2f3b5464","type":"Observation","name":"新客户注册量首次突破日均50人","aliases":["新客户注册量首次突破日均50人"],"alias_bindings":[],"identifiers":[]}

ENTITY_PROFILE {"id":"entity_4f00719703603e07346a","type":"Event","name":"全局定价策略变更","aliases":["全局定价策略变更"],"alias_bindings":[],"identifiers":[]}

ENTITY_PROFILE {"id":"entity_f85e36ae06194622425a","type":"Product"

## Cell 21 — 调用 OpenRouter 得到最终答案

这一格使用**刚才已经生成的同一份 GraphRAG context**，直接调用 `stealth/space-bunny-alpha`。

所以如果最终答案有问题，你可以准确区分：

- context 已经有答案但模型没答好 → Final Prompt / 模型问题；
- context 本身缺答案 → Retrieval / 上游 KG 问题。

最终模型被要求直接输出三段 Markdown：

1. `## 1. 结论`
2. `## 2. 推理路径`
3. `## 3. 不确定项`

不走“模型 JSON → Python 再渲染”的额外流程。
先运行 Cell 20，再运行本格；本格自行准备 API 配置，复用已有 Key 或通过密码输入框询问，不需要运行抽取/对齐。


In [41]:
def call_final_answer(question, retrieval_result):
    payload = {
        'model': MODEL,
        'messages': [
            {'role': 'system', 'content': ANSWER_PROMPT},
            {'role': 'user', 'content': retrieval_result['context']},
        ],
        'max_tokens': QA_MAX_TOKENS,
        'stream': False,
    }
    if QA_REASONING:
        payload['reasoning'] = {'effort': QA_REASONING}
    headers = {'Authorization': 'Bearer ' + OPENROUTER_API_KEY, 'Content-Type': 'application/json'}
    t0 = time.perf_counter()
    r = requests.post(
        OPENROUTER_BASE_URL.rstrip('/') + '/chat/completions',
        headers=headers, json=payload, timeout=(30, HTTP_TIMEOUT)
    )
    elapsed = time.perf_counter() - t0
    r.raise_for_status()
    body = r.json()
    answer = body.get('choices', [{}])[0].get('message', {}).get('content', '')
    usage = body.get('usage') or {}
    call_logs.append({
        'stage': 'qa', 'batch_index': 0, 'tasks': 1, 'attempt': 1, 'status': 'ok',
        'elapsed_s': round(elapsed, 3),
        'prompt_tokens': usage.get('prompt_tokens'), 'completion_tokens': usage.get('completion_tokens'),
        'total_tokens': usage.get('total_tokens'), 'cost': usage.get('cost'),
    })
    return answer, usage, elapsed

if 'QUESTION' not in globals() or 'retrieval' not in globals():
    raise RuntimeError('请先运行 Cell 20，得到同一问题的检索上下文，再运行 Cell 21。')

# 独立准备本 cell 所需环境；不运行安装、解析、抽取、对齐或构图。
from pathlib import Path
from IPython.display import display, Markdown
import importlib, sys

_query_notebook = Path.cwd() / 'evidence_kg_openrouter_debug.ipynb'
if not _query_notebook.is_file():
    raise FileNotFoundError('请在本 notebook 所在目录启动内核。')
if str(_query_notebook.parent) not in sys.path:
    sys.path.insert(0, str(_query_notebook.parent))
import interactive_kg as _query_module
importlib.reload(_query_module)
_single_runtime = _query_module.query_runtime(_query_notebook, overrides=globals())
call_logs = _single_runtime['call_logs']
RAG_WORK_DIR = _single_runtime['RAG_WORK_DIR']
if 'rag' in globals() and Path(rag.work).resolve() != RAG_WORK_DIR:
    raise RuntimeError('问答图谱已变化，请重新运行 Cell 20 获取新图谱的检索上下文，再运行 Cell 21。')
if not _single_runtime['OPENROUTER_API_KEY']:
    from getpass import getpass
    _single_runtime['OPENROUTER_API_KEY'] = getpass('OpenRouter API Key: ').strip()
if not _single_runtime['OPENROUTER_API_KEY']:
    raise ValueError('API Key 不能为空。')

answer, qa_usage, qa_elapsed = _single_runtime['call_final_answer'](QUESTION, retrieval)
print(f'QA wall time: {qa_elapsed:.2f}s')
print('usage =', qa_usage)
print('\n' + '=' * 100 + '\n')
display(Markdown(answer))

QA wall time: 61.40s
usage = {'prompt_tokens': 8410, 'completion_tokens': 3837, 'total_tokens': 12247, 'cost': 0, 'is_byok': False, 'prompt_tokens_details': {'cached_tokens': 138, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'image_tokens': 0, 'audio_tokens': 0}}




## 1. 结论
6条线索按时间为：**2025-10-13，新客户注册量首次突破日均50人（J8）→ 2026-04-08，全局定价由免费增值改为订阅制，基础版9.9元/月、高级版29.9元/月（J33）→ 2026-05-06，Cursor v1→v2导致code-assist-pro出现兼容性问题（J63）→ 2026-05-20，设计素材包V3停售并由V4替代、文案助手并入周报工具（J78）→ 2026-06-01，客户投诉激增，工单由月均45条升至128条（J93）→ 2026-06-30，Q2归因显示定价策略贡献收入下降17%、工具升级兼容性贡献下降12%，P03收入下降29%（J108）**。核心因果链是：**Cursor升级 → code-assist-pro兼容性问题及停更两周 → 投诉增加和收入下降**；**定价改为订阅制 → 新用户付费率与老用户续费率下降 → 收入下降**。两项归因合计正好对应P03收入下降29%；素材包与文案工具调整是同期发生的另一项产品组合变更，材料未显示其参与该29%的收入归因。

## 2. 推理路径
新客户注册量首次突破日均50人（来源：测试集A_运营数据记录.xlsx，“订单明细”J8）→ 2026-04-08全局定价策略变更（来源：测试集A_运营数据记录.xlsx，“订单明细”J33；测试集A_产品运营分析报告.pptx，第2页）→ 定价调整后新用户付费率由7.8%降至2.1%、续费率由82%降至63%，定价策略对收入下降贡献17%（来源：测试集A_工作室运营手册.docx，“2.1 定价策略与AB实验”）→ 2026-05-06 Cursor v1→v2（来源：测试集A_运营数据记录.xlsx，“订单明细”J63；测试集A_产品运营分析报告.pptx，第6页）→ 触发code-assist-pro兼容性问题并停更适配（来源：测试集A_运营数据记录.xlsx，“订单明细”J63；测试集A_产品运营分析报告.pptx，第6页）→ 停更期间已购用户投诉增加，工单由45条升至128条/月（来源：测试集A_产品运营分析报告.pptx，第6页；“订单明细”J93）→ 2026-06-01确认P03收入由12.8万元降至9.1万元、下降29%（来源：测试集A_产品运营分析报告.pptx，第2、6页）→ Q2归因将降幅分解为定价策略−17%和工具升级兼容性−12%（来源：测试集A_运营数据记录.xlsx，“订单明细”J108）；同期2026-05-20设计素材包V3由V4替代、文案助手并入周报工具（来源：测试集A_运营数据记录.xlsx，“订单明细”J78；测试集A_产品运营分析报告.pptx，第2页）。

## 3. 不确定项
无重要不确定项。

## Cell 22 — 可选：一次并发测试多道题

如果你已经完成构图，后面的每道 QA 彼此独立，可以 4 路并发。

把问题放进 `QUESTIONS`。每道题会先在本地完成 Lean GraphRAG retrieval，再并发调用 OpenRouter 最终回答。

这格特别适合你一次快速测 Q1/Q2/Q5 等不同题型，看：

- cross-format identity 是否走 Entity Profile / source-aware retrieval；
- Excel Sheet/列扫描是否走 Structured；
- 真正多跳因果题是否才走 weighted beam。
本格可直接运行；保留当前 `QUESTIONS` 列表。无需运行 Cell 9 或构图部分，调用日志与 Cell 23、Cell 24 共享。


In [47]:
# 独立准备本 cell 所需环境；不运行安装、解析、抽取、对齐或构图。
from pathlib import Path
from IPython.display import display, Markdown
import importlib, sys

_query_notebook = Path.cwd() / 'evidence_kg_openrouter_debug.ipynb'
if not _query_notebook.is_file():
    raise FileNotFoundError('请在本 notebook 所在目录启动内核。')
if str(_query_notebook.parent) not in sys.path:
    sys.path.insert(0, str(_query_notebook.parent))
import interactive_kg as _query_module
importlib.reload(_query_module)
_qa_runtime = _query_module.query_runtime(_query_notebook, overrides=globals())
call_logs = _qa_runtime['call_logs']
RAG_WORK_DIR = _qa_runtime['RAG_WORK_DIR']
if not _qa_runtime['OPENROUTER_API_KEY']:
    from getpass import getpass
    _qa_runtime['OPENROUTER_API_KEY'] = getpass('OpenRouter API Key: ').strip()
if not _qa_runtime['OPENROUTER_API_KEY']:
    raise ValueError('API Key 不能为空。')

from concurrent.futures import ThreadPoolExecutor, as_completed

_selected = _qa_runtime['RAG_GRAPH_SELECTION']
print(f'[图谱] source={_selected["source"]} backend={_selected["backend"]} '
      f'nodes={_selected["nodes"]} facts={_selected["facts"]} path={RAG_WORK_DIR}')

QUESTIONS = [
    "2026年第二季度，明创工坊的明星产品\"AI代码助手插件\"季度收入从12.8万元降至9.1万元，下降29%。请按时间顺序列出Q2期间与该产品收入下降相关的全部变更事件，并说明每个事件与收入下降的因果关系。注意：相关线索分散在Excel（含隐藏的备注列）、Word、PDF、PPT等多种文档中，且该产品在不同文档中使用不同名称。",
    "\"AI代码助手插件\"这一产品在PDF、Word、Excel、PPT、图片5种格式文档中分别使用了什么名称？每种格式中包含哪些仅在该格式中可见的独有信息？请逐一列出。",
    "客户\"程序员老王\"购买了\"AI代码助手插件\"的年度订阅。请给出从该客户到产品收入下降的完整关联链路，包括：客户→产品→依赖的AI工具→定价策略变更→续费结果。标注每跳的信息来源。",
    "PDF文档中的产品定价规格表（T1）在跨页续表中省略了\"定价\"和\"适用平台\"两列。对于续表中出现的产品，请说明如何通过其他文档（Word/Excel/PPT）还原这些缺失的定价和平台信息，并给出至少2个具体产品的还原路径。",
    "Excel\"订单明细\"Sheet的备注列中隐藏了6条关键线索（分布在不同的行中）。请找出全部6条隐藏线索，并按时间顺序串联成一条完整的事件链，说明这些事件之间的因果关系。",
    "\"设计素材包V3\"被\"设计素材包V4\"替代。请说明：(1)替代的原因是什么？(2)V4覆盖了V3的哪些内容？(3)V4上线后的用户评分反馈如何？(4)这个替代决策记录在哪些文档中？",
    "2026年4月8日的定价策略变更（免费增值→订阅制）对哪些产品和客户影响最大？请列出受影响最大的3个产品及其影响数据，并说明哪些产品不受影响及原因。",
    "Cursor v3即将发布，\"AI代码助手插件\"面临什么风险？请根据Word文档中的风险预警说明，分析可能的影响路径和张明的应对建议。",
    "编号\"RV-2026-0412\"在PPT中出现在两个不同的场景中。请说明这两个场景分别是什么，该编号被重复使用是否会造成歧义？在构建知识图谱时如何处理这种编号重复的情况？",
    "PDF文档中的AI工具使用对照表（T2）的续表清空。请根据首页表头和续表数据，还原正确的列对应关系，并说明判断依据。同时，该表中哪些AI工具与\"AI代码助手插件\"存在直接或间接的关联？",
]

def answer_one_question(index, q):
    local_rag = _qa_runtime['LeanGraphRAG'](RAG_WORK_DIR, max_hops=3, beam_width=20, context_chars=18_000)
    ret = local_rag.retrieve(q)
    ans, usage, elapsed = _qa_runtime['call_final_answer'](q, ret)
    return {
        'index': index, 'question': q, 'answer': ans, 'usage': usage,
        'elapsed_s': elapsed, 'query_plan': ret['query_plan'], 'stats': ret['stats']
    }

if QUESTIONS:
    multi_results = []
    with ThreadPoolExecutor(max_workers=min(_qa_runtime['QA_CONCURRENCY'], len(QUESTIONS))) as pool:
        futs = {pool.submit(answer_one_question, i, q): (i, q) for i, q in enumerate(QUESTIONS, 1)}
        for fut in as_completed(futs):
            res = fut.result()
            multi_results.append(res)
            print(f'Q{res["index"]} finished in {res["elapsed_s"]:.1f}s '
                  f'| plan={res["query_plan"]["intent"]} '
                  f'mode={res["stats"]["retrieval_mode"]} '
                  f'structured={res["stats"]["structured"]} '
                  f'scope={res["stats"]["structured_scope_matches"]} '
                  f'augmentation={res["stats"]["structured_selected_for_augmentation"]} '
                  f'table_units={res["stats"].get("table_units", 0)} '
                  f'table_columns={res["stats"].get("table_columns", 0)} '
                  f'table_rows={res["stats"].get("table_context_rows", 0)}/{res["stats"].get("table_rows", 0)} '
                  f'timeline={res["stats"].get("timeline_context_events", 0)}/{res["stats"].get("timeline_events", 0)} '
                  f'paths={res["stats"]["paths"]}')
    multi_results.sort(key=lambda x: x['index'])
    for res in multi_results:
        print('\n' + '=' * 110)
        print(f'Q{res["index"]}: {res["question"]}')
        display(Markdown(res['answer']))
else:
    print('QUESTIONS 为空；需要批量测试时填入问题列表后重跑本格。')

[图谱] source=latest backend=native nodes=286 facts=1344 path=C:\Users\Agesen\Downloads\Task4\github-release\evidence-kg-graphrag\kg_debug_run_q5q10_fix\work
Q2 finished in 17.1s | plan=cross_format_identity mode=ranked structured=24 scope=0 augmentation=0 table_units=0 table_columns=0 table_rows=0/0 timeline=0/0 paths=0
Q3 finished in 17.7s | plan=factual mode=ranked structured=24 scope=0 augmentation=0 table_units=0 table_columns=0 table_rows=0/0 timeline=0/0 paths=6
Q4 finished in 22.5s | plan=structured_lookup mode=ranked structured=24 scope=0 augmentation=0 table_units=2 table_columns=7 table_rows=36/36 timeline=0/0 paths=6
Q6 finished in 20.0s | plan=causal mode=ranked structured=24 scope=0 augmentation=0 table_units=0 table_columns=0 table_rows=0/0 timeline=0/0 paths=6
Q7 finished in 24.6s | plan=causal mode=ranked structured=24 scope=0 augmentation=0 table_units=0 table_columns=0 table_rows=0/0 timeline=0/0 paths=6
Q1 finished in 51.5s | plan=causal mode=ranked structured=24 scop

## 1. 结论
按时间顺序，相关事件为：①**2026-04-08** code-assist-pro（AI代码助手插件、P03）从免费增值模式改为订阅制，4月12日完成方案评审并落实到P03付费用户；订单归因显示该定价策略贡献Q2收入下降的**17%**。②**2026年5—6月** GitHub Copilot免费策略上线，先降低P03的付费价值，随后在6月分流用户；该事件进一步削弱付费转化，但材料未给出可与17%、12%相加的独立贡献比例。③**2026-05-06** Cursor v2发布并变更代码补全API接口签名，P03原有分析模块需要重写且无法正常发版，覆盖全部用户。④**2026-05-08** P03暂停v2.2并回退v2.1，但兼容问题仍在持续。⑤**2026-05-15** 启动紧急修复；5月6日至20日适配停更，已购用户投诉由45条/月升至128条/月。⑥**2026-05-20** Cursor v2.3发布并解决P03兼容问题，结束持续约两周的适配中断。订单归因显示工具升级兼容性问题贡献收入下降的**12%**。两项已量化贡献合计**29%**，与收入从12.8万元降至9.1万元、约下降28.9%相符；直接原因仍是**订阅制调整和Cursor升级兼容故障的双重影响**。

## 2. 推理路径
“AI代码助手插件” →（同一产品P03/code-assist-pro；来源：测试集A_工作室运营手册.docx，第2.1节；测试集A_产品规格与定价文档.pdf，第5、8、14—15页；测试集A_运营数据记录.xlsx，“产品变更记录”A15:I21；测试集A_产品运营分析报告.pptx，第2、3、6、7页）→ 2026-04-08定价调整 →（免费增值→订阅制，影响P03付费用户；来源：测试集A_工作室运营手册.docx，第2.1节；测试集A_运营数据记录.xlsx，“产品变更记录”A15:I15、A17:I17；测试集A_产品规格与定价文档.pdf，第5页）→ 订单归因定价策略对Q2收入下降贡献-17%（来源：测试集A_运营数据记录.xlsx，“订单明细”A80:J116）→ 2026年5月Copilot免费策略降低付费价值、2026年6月分流P03用户（来源：测试集A_运营数据记录.xlsx，“AI工具使用统计”A1:H26）→ 2026-05-06 Cursor v2接口变化 →（P03模块需重写、无法正常发版；来源：测试集A_产品规格与定价文档.pdf，第8、15页）→ 2026-05-08回退v2.1（来源：测试集A_产品运营分析报告.pptx，第7页；测试集A_运营数据记录.xlsx，“产品变更记录”A20:H20）→ 2026-05-15紧急修复但投诉增加、适配持续停更（来源：测试集A_产品运营分析报告.pptx，第7页；测试集A_运营数据记录.xlsx，“产品变更记录”A21:I21）→ 2026-05-20 v2.3解决兼容问题（来源：测试集A_产品运营分析报告.pptx，第7页）→ 订单归因兼容性问题贡献收入下降-12%（来源：测试集A_运营数据记录.xlsx，“订单明细”A80:J116）→ 收入记录为12.8万元降至9.1万元、续费率由82%降至63%（来源：测试集A_产品运营分析报告.pptx，第7页；测试集A_运营数据记录.xlsx，“收入与续费率统计”I19）。6月1日收入分析及6月25日季度复盘只是确认上述“双重影响”，没有形成新的独立原因事件（来源：测试集A_运营数据记录.xlsx，“产品变更记录”A26:I26、A31:H31）。

## 3. 不确定项
Copilot免费策略虽然被明确记录为降低付费价值并分流用户，但没有给出独立的收入下降贡献比例，不能确定它是否已部分包含在“定价策略-17%”中；此外，2026-06-15材料使用的是“记录日期”，且“Cursor v2.3于5月20日解决兼容问题”与“P03 v2.3于6月15日适配Cursor v2”可能分别指底层工具和插件版本，材料不足以把6月15日认定为收入下降原因。


Q2: "AI代码助手插件"这一产品在PDF、Word、Excel、PPT、图片5种格式文档中分别使用了什么名称？每种格式中包含哪些仅在该格式中可见的独有信息？请逐一列出。


## 1. 结论
“AI代码助手插件”（实体：**code-assist-pro**）的跨格式名称及各自独有信息如下：

1. **PDF：`code-assist-pro`**  
   独有信息：**产品定价规格矩阵**。

2. **Word：`代码助手`**  
   独有信息：**兼容性问题与风险预警**，包括深度依赖 Cursor，以及 Cursor v2 升级造成的功能失效、发布受阻、用户体验下降和用户流失风险。

3. **Excel：`代码助手插件P03`**  
   独有信息：**收入和续费率数据**。

4. **PPT：`智能编程插件`**  
   独有信息：**迭代记录号 `IT-2026-008` 和评审编号 `RV-2026-0412`**。

5. **图片：`AI编程工具`**  
   独有信息：**产品架构图中的产品依赖关系和姊妹产品连线**。

## 2. 推理路径
- “AI代码助手插件” →（同一产品实体，标准名称）→ `code-assist-pro` →（5种跨格式别名绑定）→ PDF `code-assist-pro`、Word `代码助手`、Excel `代码助手插件P03`、PPT `智能编程插件`、图片 `AI编程工具`（来源：**测试集A_产品规格与定价文档.pdf**，第15页，表2，第2行各格式称呼列；**测试集A_工作室运营手册.docx**，“6. 产品名称对照表”，表2第2行）。
- `code-assist-pro` →（PDF独有）→ **定价规格矩阵**；→（Word独有）→ **兼容性问题与风险预警**；→（Excel独有）→ **收入和续费率数据**；→（PPT独有）→ **`IT-2026-008`、`RV-2026-0412`**；→（图片独有）→ **产品依赖关系及姊妹产品连线**（来源：**测试集A_产品规格与定价文档.pdf**，第16页，表9后的说明）。
- Word 独有风险的具体表现可由 Cursor v2 接口和补全机制变化、代码分析模块失效、无法正常发布更新及影响续费意愿等记录核实（来源：**测试集A_工作室运营手册.docx**，“3.2 产品兼容性与更新”，第24—39段）。
- Excel 的收入和续费率字段实际出现在“收入与续费率统计”Sheet，包含月收入、付费用户数、新增用户、流失用户、续费率和客单价（来源：**测试集A_运营数据记录.xlsx**，“收入与续费率统计”Sheet，第1—31行）。

## 3. 不确定项
无重要不确定项。


Q3: 客户"程序员老王"购买了"AI代码助手插件"的年度订阅。请给出从该客户到产品收入下降的完整关联链路，包括：客户→产品→依赖的AI工具→定价策略变更→续费结果。标注每跳的信息来源。


...

## 1. 结论
Cursor v3预计于2026年Q3（9—10月）发布，可能再次调整API，AI代码助手插件（Word称“代码助手”、code-assist-pro）需要重新适配，预计周期为2—3周，风险评级为高。可能的影响路径是：版本接口变化 → 插件适配与停更 → 用户体验下降、续费意愿降低 → 续费率跌破50% → 产品面临持续收入风险，并可能需要调整产品方向。张明的应对建议是预留v3适配开发时间，并在v3 beta阶段提前完成适配，以压缩停更窗口并降低续费损失。

## 2. 推理路径
AI代码助手插件 →(深度依赖)→ Cursor编程环境（来源：测试集A_工作室运营手册.docx，§3.2，段落24—39）→(版本升级导致接口和补全机制变化)→ Cursor v2曾使代码分析模块失效（来源：测试集A_工作室运营手册.docx，§3.2，段落24—39）→(同类变化可能再次发生)→ Cursor v3可能变更API、插件需要再次适配（来源：测试集A_工作室运营手册.docx，§5.2，段落61—75）→(适配期间若再次停更)→ 续费率可能降至50%以下并形成持续收入风险（来源：测试集A_工作室运营手册.docx，§5.2，段落61—75）。张明负责该风险事项（来源：测试集A_运营数据记录.xlsx，“产品变更记录”Sheet第29行、H29）→(预留开发时间)→ 在Cursor v3 beta阶段提前适配，预计投入2—3周（来源：测试集A_工作室运营手册.docx，§5.2，段落61—75）。

## 3. 不确定项
Cursor v3的发布窗口及API变更幅度仍是预估，实际适配周期取决于最终变更范围；若2026年Q3续费率低于50%，文档仅明确提出应考虑调整产品方向，未给出具体调整方案。


Q9: 编号"RV-2026-0412"在PPT中出现在两个不同的场景中。请说明这两个场景分别是什么，该编号被重复使用是否会造成歧义？在构建知识图谱时如何处理这种编号重复的情况？


## 1. 结论
编号“RV-2026-0412”被用于两个场景：一是2026年4月12日的“定价策略变更方案评审”，评审结论为有条件通过，关联变更编号为CHG-014；二是2026年5月25日的“设计素材包V4质量反馈处理单”。单独使用该编号会造成歧义，因为它不是唯一标识；但结合记录类型、日期和业务对象后可以区分。构建知识图谱时，应将两个业务实体分别建档，不按该编号合并；把“RV-2026-0412”建模为被多个实体复用的标识，并增加实体类型、日期及来源等限定，例如使用“评审编号:RV-2026-0412@2026-04-12”和“处理单编号:RV-2026-0412@2026-05-25”，同时保留 `reuses_identifier_of` 关系及证据来源。

## 2. 推理路径
“RV-2026-0412” →(评审编号)→“定价策略变更方案评审”，记录日期为2026-04-12、评审人为张明、关联变更为CHG-014（来源：测试集A_产品运营分析报告.pptx，Slide 7，Rounded Rectangle 4）→(评审对象)→“code-assist-pro/智能编程插件”。另一路径为“RV-2026-0412” →(处理单编号)→“设计素材包V4质量反馈处理单”，日期为2026-05-25（来源：测试集A_产品运营分析报告.pptx，Slide 7，Rounded Rectangle 4；Slide 8，TextBox 10）→(重复使用)→“定价策略变更评审”。Slide 8明确说明该编号用于定价策略变更评审，并指出同一编号在5月25日亦用于V4质量反馈处理单（来源：测试集A_产品运营分析报告.pptx，Slide 8，TextBox 10）；Slide 7的备注直接标明“编号重复使用”（来源：测试集A_产品运营分析报告.pptx，Slide 7，Rounded Rectangle 4）。因此，图谱应保留两个事件节点及其重复标识关系，而不是将二者错误合并。

## 3. 不确定项
“V4质量反馈处理单”与“设计素材包V4质量反馈处理单”在不同证据中的名称范围略有差异，材料未提供显式的别名绑定；这不影响“同一编号被两个场景复用”的结论，但若需严格确认处理单对应的完整产品名称，应进一步核验原始业务记录。


Q10: PDF文档中的AI工具使用对照表（T2）的续表清空。请根据首页表头和续表数据，还原正确的列对应关系，并说明判断依据。同时，该表中哪些AI工具与"AI代码助手插件"存在直接或间接的关联？


## 1. 结论
T2续表应继续沿用首页的8列顺序：**工具名称｜用途分类｜主要使用场景｜调用频次/月｜月度成本｜版本｜依赖产品｜备注**。例如“Cursor月调用量—8万次—v2—v1→v2迁移后统计”分别落在第1、4、6、8列，其余列保留“—”；“162美元/月”落在第5列，“code-assist-pro”落在第7列。与“AI代码助手插件”（即 **code-assist-pro**）有关联的AI工具共有：**Cursor为直接关联，GitHub Copilot为间接关联**。其中“Cursor月调用量”和“Cursor v2适配进度”只是Cursor的统计或进度记录，“Copilot月调用量”和“Copilot免费政策”只是GitHub Copilot的统计或政策记录，不应算作独立工具。

## 2. 推理路径
T2首页表头 →(定义8列及顺序)→ 工具名称、用途分类、主要使用场景、调用频次/月、月度成本、版本、依赖产品、备注（来源：测试集A_产品规格与定价文档.pdf，第7页表1，表头/第2—15行）→(续表schema明确继承第7页表头)→ 第8页续表使用同一8列结构（来源：测试集A_产品规格与定价文档.pdf，第8页表1，TABLE_SCHEMA）→(数据位置交叉验证)→ “8万次”“3万次”“2千次”属于调用频次/月，“162美元/月”和“约18万元/月”属于月度成本，“v2/v1.5/v2024”属于版本，“v1→v2迁移后统计”“占支出35%”等属于备注（来源：同PDF第8页表1，第5—13行）。具体对应为：**Cursor月调用量—8万次—v2—v1→v2迁移后统计｜Copilot月调用量—3万次—v1.5—免费额50000次｜Figma AI月调用量—2千次—v1—设计素材V4开发｜Studio总成本统计—162美元/月—占支出35%｜Studio总收入统计—约18万元/月—Q2前约15万元/月｜Cursor v2适配进度—已完成2周—v2—code-assist-pro—5月20日恢复更新｜Copilot免费政策—50→200次/月—v1.5—影响P03付费价值｜Tool链覆盖率—18种—全产品线—编程3+设计3+文案2+其它10｜自动化节省工时—约80小时/月—Zapier+Make合计**（来源：同PDF第8页表1，第5—13行）。关联链为：**AI代码助手插件 →(别名)→ code-assist-pro**（来源：测试集A_产品规格与定价文档.pdf，第15页表2第2行；实体别名校验）→(依赖产品)→ **Cursor**（来源：同PDF第7页表1第2行、第8页表1第10行），因此是直接关联；**GitHub Copilot →(代码补全工具，且免费政策影响P03付费价值)→ P03/code-assist-pro**（来源：同PDF第7页表1第3行、第8页表1第6及11行；第15页表2第2行确认P03身份），因此属于间接关联。Excel记录还明确把GitHub Copilot的关联产品写为code-assist-pro（来源：测试集A_运营数据记录.xlsx，“AI工具使用统计”第11—12行，F11:F12）。

## 3. 不确定项
无重要不确定项。

## Cell 23 — 汇总整次 OpenRouter 性能

最后把所有模型调用汇总成表。

最值得看的指标：

- `elapsed_s`：单请求墙钟时间；
- `prompt_tokens`：Alignment 如果仍然很高，说明 candidate profile 还太肥；
- `completion_tokens`：Extraction 如果仍然动辄上万，说明模型还在重复 evidence / 原文；
- `stage` 聚合后的总 token 和平均耗时；
- failed request / retry 情况。

**并发情况下不要把每个请求 elapsed 直接相加当总墙钟。**它反映的是 provider 单请求延迟；真正总耗时应该看每个阶段从开始到结束的 wall time。
本格可在 Cell 21、22 或 24 后运行；没有请求时显示空日志，不会调用模型或构图。


In [48]:
# 独立准备本 cell 所需环境；不运行安装、解析、抽取、对齐或构图。
from pathlib import Path
from IPython.display import display, Markdown
import importlib, sys

_query_notebook = Path.cwd() / 'evidence_kg_openrouter_debug.ipynb'
if not _query_notebook.is_file():
    raise FileNotFoundError('请在本 notebook 所在目录启动内核。')
if str(_query_notebook.parent) not in sys.path:
    sys.path.insert(0, str(_query_notebook.parent))
import interactive_kg as _query_module
importlib.reload(_query_module)
_metrics_runtime = _query_module.query_runtime(_query_notebook, overrides=globals())
call_logs = _metrics_runtime['call_logs']
RAG_WORK_DIR = _metrics_runtime['RAG_WORK_DIR']

import pandas as pd
import json

log_df = pd.DataFrame(call_logs)
if len(log_df):
    display(log_df)
    print('\n=== 按 stage 聚合 ===')
    agg = log_df.groupby('stage').agg(
        requests=('stage', 'size'),
        avg_latency_s=('elapsed_s', 'mean'),
        p95_latency_s=('elapsed_s', lambda s: s.quantile(0.95)),
        prompt_tokens=('prompt_tokens', 'sum'),
        completion_tokens=('completion_tokens', 'sum'),
        total_tokens=('total_tokens', 'sum'),
    ).reset_index()
    display(agg)

    metrics_path = _metrics_runtime['RUN_ROOT'] / 'openrouter_metrics.json'
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    metrics_path.write_text(json.dumps(call_logs, ensure_ascii=False, indent=2), encoding='utf-8')
    print('完整调用日志已保存：', metrics_path)
else:
    print('当前还没有 OpenRouter 调用日志。')

,stage,batch_index,tasks,attempt,status,elapsed_s,prompt_tokens,completion_tokens,total_tokens,cost,raw_path
0,extract,6,4,1,ok,16.873,3409,2055,5464,0,C:\Users\Agesen\Downloads\Task4\github-release...
1,extract,7,5,1,ok,23.636,3599,3747,7346,0,C:\Users\Agesen\Downloads\Task4\github-release...
2,extract,4,5,1,ok,30.403,6116,3061,9177,0,C:\Users\Agesen\Downloads\Task4\github-release...
3,extract,1,5,1,ok,45.413,21182,5552,26734,0,C:\Users\Agesen\Downloads\Task4\github-release...
4,extract,2,5,1,ok,46.507,8026,4338,12364,0,C:\Users\Agesen\Downloads\Task4\github-release...
...,...,...,...,...,...,...,...,...,...,...,...
60,qa,0,1,1,ok,51.452,6166,3547,9713,0,NaN
61,qa,0,1,1,ok,16.088,7448,684,8132,0,NaN
62,qa,0,1,1,ok,43.688,8491,2036,10527,0,NaN
63,qa,0,1,1,ok,18.149,8038,792,8830,0,NaN



=== 按 stage 聚合 ===


,stage,requests,avg_latency_s,p95_latency_s,prompt_tokens,completion_tokens,total_tokens
0,align,14,8.589143,14.41375,122960,7292,130252
1,extract,14,38.209929,88.01030,77739,56143,133882
2,qa,37,30.855784,51.86900,275293,71075,346368


完整调用日志已保存： C:\Users\Agesen\Downloads\Task4\github-release\evidence-kg-graphrag\kg_debug_run_q5q10_fix\openrouter_metrics.json


# 结果应该怎么判断？

跑完以后，可以用下面这套非常直接的判断标准：

### Parse / Prepare 健康

- 所有 corpus 文档 `status=success`；
- DocumentUnit 中能看到原始关键内容和 locator；
- Excel 普通结构走 deterministic，自由文本走 semantic overlay；
- 不应该因为某个 Unit 没“完美 evidence”而停流程。

### Extraction 健康

- 模型输出主要是短 JSON delta；
- completion token 明显低于旧版那种每批一两万甚至两万多 token 的水平；
- `model unresolved` 只是真歧义，不是格式审计垃圾；
- unknown predicate / 普通数值不会把整 task 弄失败。

### Alignment 健康

- 明显同一稳定实体能合；
- 不同版本/强 ID 冲突不会误合；
- 每个 task 只输出 decision；
- cluster 数明显小于 raw mention 数；
- `UNCERTAIN` 不需要追求 0，只要别因为“上下文不同”过度拆分。

### GraphRAG 健康

- Sheet/列问题：`structured_hits` 应该是主力；
- 跨格式名称题：`query_plan.intent` 应是 `cross_format_identity`，`entity_profiles` 应包含 aliases / alias_bindings，`paths` 通常应为 0；
- 多跳因果题：才应该出现 weighted beam graph paths；
- `raw_units` 应覆盖题目明确要求的来源格式；
- 最重要：**最终 context 里必须真的包含你希望模型回答的证据。**

### Final Answer 健康

必须直接出现：

```text
## 1. 结论
## 2. 推理路径
## 3. 不确定项
```

只有会改变主要结论的关键缺失/冲突才进入“不确定项”，不要再输出大量防御性免责声明。

## Cell 24 — 交互问答：自动选择已有 GraphRAG

运行本格，填写 API Key、输入问题并点击“提问”。

运行时自动寻找当前 `PARSER_BACKEND` 最新的完整图谱，找不到时回退之前的 native 图谱。上方 `[图谱]` 日志显示实际选择的路径、后端、latest/fallback 和规模。

此入口只查询已有图谱，不会自动解析、抽取、对齐或构图。新图谱生成后，重新运行本格即可自动切换到它。


In [ ]:
from pathlib import Path
import importlib, sys

_chat_notebook = Path.cwd() / 'evidence_kg_openrouter_debug.ipynb'
if not _chat_notebook.is_file():
    raise FileNotFoundError('请在本 notebook 所在目录启动内核，或将 _chat_notebook 改为它的完整路径。')
if str(_chat_notebook.parent) not in sys.path:
    sys.path.insert(0, str(_chat_notebook.parent))
import interactive_kg as _interactive_kg
importlib.reload(_interactive_kg)

# 与单题/并发入口使用同一套自动选择规则；不触发构图。
_chat_runtime = _interactive_kg.query_runtime(_chat_notebook, overrides=globals())
RAG_WORK_DIR = _chat_runtime['RAG_WORK_DIR']
call_logs = _chat_runtime['call_logs']
_selected = _chat_runtime['RAG_GRAPH_SELECTION']
print(f'[图谱] source={_selected["source"]} backend={_selected["backend"]} '
      f'nodes={_selected["nodes"]} facts={_selected["facts"]} path={RAG_WORK_DIR}')
KG_CHAT = _interactive_kg.InteractiveKGSession(
    _chat_notebook, overrides=_chat_runtime, graph_work_dir=RAG_WORK_DIR,
)
KG_CHAT_UI = _interactive_kg.display_chat(KG_CHAT)


[图谱] source=latest backend=native nodes=286 facts=1344 path=C:\Users\Agesen\Downloads\Task4\github-release\evidence-kg-graphrag\kg_debug_run_q5q10_fix\work


In [38]:
from lean_graphrag import LeanGraphRAG

rag = LeanGraphRAG(
    r"C:\Users\Agesen\Downloads\Task4\github-release\evidence-kg-graphrag\examples\prebuilt\work",
    context_chars=18000,
)

q = 'Excel"订单明细"Sheet的备注列中隐藏了6条关键线索（分布在不同的行中）。请找出全部6条隐藏线索，并按时间顺序串联成一条完整的事件链，说明这些事件之间的因果关系。'

ret = rag.retrieve(q)

print("=== QUERY PLAN ===")
print(ret["query_plan"])

print("\n=== STRUCTURED HITS ===")
print("count =", len(ret["structured_hits"]))

for hit in ret["structured_hits"]:
    print(
        hit.get("locator"),
        "| field =", hit.get("field"),
        "| value =", hit.get("value")
    )

print("\n=== AUGMENTATION QUERY ===")
print(ret.get("augmentation_query"))

=== QUERY PLAN ===
{'intent': 'structured_enumeration', 'structured': True, 'cross_format_identity': False, 'table_reconstruction': False, 'retrieval_mode': 'exhaustive', 'requested_formats': ['excel'], 'list_all': True, 'causal': True, 'comparison': False, 'rank': False, 'timeline': True, 'expected_count': 6, 'sheet_hints': ['订单明细'], 'column_hints': ['备注'], 'years': [], 'months': [], 'quarters': [], 'keywords': ['excel', 'Excel"订单明细"', '备注列', '隐藏', '6条关键线索', '分布', '不同', '6条隐藏线索', '并按时间顺序串联成一条完整', '事件链', '这些事件之间', '关系'], 'relation_weights': {'reported_cause': 3.2, 'purchased': 2.0, 'affects': 2.5, 'has_observation': 1.8, 'depends_on': 1.3, 'upgraded_to': 1.2, 'replaced_by': 1.1}}

=== STRUCTURED HITS ===
count = 8
{'sheet': '订单明细', 'sheet_state': 'visible', 'row_start': 1, 'row_end': 40, 'cell_range': 'A1:J40', 'row': 8, 'cell': 'J8'} | field = 备注 | value = 新客户注册量首次突破日均50人
{'sheet': '订单明细', 'sheet_state': 'visible', 'row_start': 1, 'row_end': 40, 'cell_range': 'A1:J40', 'row': 18, 'cel